# Which components build the comparator?

Qwen2.5-7B-Instruct, `max(a, b)` one-shot prompt, both cases (`n1 larger`, `n2 larger`). Both
numbers are in the L14 residual at the second-number position $n_2$: $\mathbf{u}$ = H14 DAS 1D
(inside head L14.H14's output), $\mathbf{v}_2$ = L13 PC1 @ $n_2$. Directions are loaded from
`results/sharedrep_2digit.npz`; nothing is retrained here.

A sub-block is **frozen** by pinning its output at $n_2$ to the value it takes on the unpatched
corrupted run, so the upstream patches cannot reach the rest of the network through it:

$$\mathrm{out}_\ell[n_2] := \mathrm{out}^{\mathrm{corr}}_\ell[n_2]$$

Sections: freeze machinery · sanity check · patch × freeze grid · L14–L16 sweep.

In [ ]:
import os, json, time
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

import data_numberreps as D
import data_sharedrep as S
from data_numberreps import (build_triples, make_batch, regime_positions, pos_anchors,
                             posrec_and_iia, _slice_batch,
                             CASE_SLOT, SEED, LAYER, REGIMES, EVAL_BS, N_EVAL, device)

RESULTS_DIR, FIGS_DIR = 'results', 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)

REGIME = '2digit'
R  = dict(np.load(f'{RESULTS_DIR}/sharedrep_{REGIME}.npz'))
MJ = json.load(open(f'{RESULTS_DIR}/sharedrep_meta.json'))['regimes'][REGIME]
M  = MJ['meta']

h14_das_np  = R['h14_das_dir'] / np.linalg.norm(R['h14_das_dir'])      # u,  inside L14.H14's output
l13_pc1b_np = R['l13_pc1b_dir'] / np.linalg.norm(R['l13_pc1b_dir'])    # v2, L13 PC1 @ n2

print(f"{REGIME}: L{M['l13_layer']} -> L{M['h_layer']}.H{M['h_head']}   "
      f"n1 tok {M['pos_first']}, n2 tok {M['pos_second']}   eval n={M['n_eval']}  case '{M['case']}'")
for nm in ['H14_DAS', 'L13_PC1_atn2', 'H14_DAS_and_L13_PC1_atn2']:
    print(f"saved reference {nm:>26}: posrec {R[f'cond_posrec_{nm}'].mean():6.3f}   "
          f"IIA {R[f'cond_iia_{nm}'].mean():.3f}")

## Plot style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 16, 13, 13, 17
DPI, SAVE = 300, True

# Same palette as makefig_sharedreps.ipynb.
COL = {'full': '#2166AC',   # blue   -> MLP freezes / arm 1
       'das':  '#C97B00',   # amber  -> attention freezes
       'pc1':  '#C06A9B',   # orchid -> L13 v2 arm
       'h14':  '#117777',   # teal   -> H14 u arms
       'ceil': '#6E6E6E'}   # grey   -> unfrozen reference
JOINT_HATCH = '//'          # joint (H14 + L13) conditions: same colour, hatched

U, V2 = r'$\mathbf{u}$', r'$\mathbf{v}_2$'


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3, axis='y')
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/comparatorfig_{name}.png', dpi=DPI, bbox_inches='tight')


# ---- run mode ---------------------------------------------------------------------------------
# Every expensive block writes one .npz. When they are all present nothing below needs the model,
# so the notebook runs for plotting alone: no GPU, a few seconds. Set PLOT_ONLY = False by hand to
# force the compute path back on; individual blocks also keep their own FORCE / FORCE_GRID flags.
# `checks_` is deliberately not in this list: nothing plots from it, and gating on it would make
# the first save flip PLOT_ONLY to True and so stop cells 9 and 11 ever recording their numbers.
RESULT_FILES = {k: f'comparator_{k}{REGIME}.npz' for k in
                ('', 'atp_', 'topk_', 'inject_', 'tune_', 'rf_', 'push_', 'mlpdas_', 'das15_',
                 'connect_', 'bound_', 'shared_')}
_missing = [f for f in RESULT_FILES.values() if not os.path.exists(f'{RESULTS_DIR}/{f}')]
PLOT_ONLY = not _missing

print('run mode:', 'PLOT ONLY -- every result is cached, the model is not loaded' if PLOT_ONLY
      else f'COMPUTE -- {len(_missing)} result file(s) missing: ' + ', '.join(_missing))

## Load the model

Separate cell — everything above runs without it.

In [ ]:
low, high, gap = (REGIMES[REGIME][k] for k in ('low', 'high', 'gap'))

if PLOT_ONLY:
    # The token layout and model dimensions are all recorded in the saved meta, so the figures
    # below need neither the weights nor the tokenizer.
    T, POS_A, POS_B = M['T'], M['pos_first'], M['pos_second']
    d_model, head_dim = M['d_model'], M['head_dim']
    print(f'plot-only: model not loaded; layout from the saved meta  '
          f'T={T}  POS_A={POS_A}  POS_B={POS_B}  d_model={d_model}')
else:
    model, tokenizer = D.load_model()
    S.init_head_machinery()
    d_model, head_dim = D.d_model, D.head_dim

    T, (POS_A, POS_B), _n_tok, _p = regime_positions(low, high)
    assert (POS_A, POS_B) == (M['pos_first'], M['pos_second']), \
        'token layout must match the saved run'

    Q_h14  = torch.tensor(h14_das_np.astype(np.float32),  device=device).reshape(-1, 1)
    P_l13b = torch.tensor(l13_pc1b_np.astype(np.float32), device=device).reshape(-1, 1)
    print(f'T={T}  POS_A={POS_A}  POS_B={POS_B}  d_model={d_model}  head_dim={head_dim}')

## Freeze machinery

`out[:, n2, :] := out_corrupted[:, n2, :]` on a decoder layer's `mlp` or `self_attn`. Attention
returns `(hidden, weights)`, the MLP a bare tensor.

In [ ]:
FREEZE_POS = POS_B          # freeze at the patched token only


def block(site):
    """('mlp'|'attn', layer) -> the module whose output is frozen."""
    kind, layer = site
    m = D.model.model.layers[layer]
    return m.mlp if kind == 'mlp' else m.self_attn


def make_freeze_hook(frozen, pos):
    """Overwrite this sub-block's output at `pos` with `frozen` (B, d_model)."""
    def _hook(mod, inp, out, _pos=pos, _f=frozen):
        hs = out[0] if isinstance(out, tuple) else out
        hs_new = hs.clone()
        hs_new[:, _pos, :] = _f.to(hs.dtype)
        return (hs_new,) + out[1:] if isinstance(out, tuple) else hs_new
    return _hook


def capture_corrupted(sites, ids, pos, bs=EVAL_BS):
    """{site: (n, d_model)} sub-block output at `pos` on the corrupted run with no intervention.

    One pass captures every site, so all the freezes come from the same forward pass.
    """
    store, chunks = {}, {s: [] for s in sites}

    def mk(site):
        def _hook(mod, inp, out):
            hs = out[0] if isinstance(out, tuple) else out
            store[site] = hs[:, pos, :].detach().float()
        return _hook

    handles = [block(s).register_forward_hook(mk(s)) for s in sites]
    try:
        for lo in range(0, ids.shape[0], bs):
            with torch.no_grad():
                _ = D.model(ids[lo:lo + bs])
            for s in sites:
                chunks[s].append(store.pop(s))
    finally:
        for h in handles:
            h.remove()
    return {s: torch.cat(v, dim=0) for s, v in chunks.items()}


def eval_spec(spec, bat, sources, pos, freeze=(), frozen=None, bs=EVAL_BS):
    """Per-example position recovery and IIA for one (patch spec, freeze set), minibatched.

    `spec` is `data_sharedrep.build_handles`'s format: {'resid': [(layer, kind, P)],
    'head': (head, P)}. The freeze hooks are registered after the patch hooks, so within a layer
    they see the patched input and still emit the corrupted output.
    """
    recs, iias = [], []
    for lo in range(0, bat['n'], bs):
        hi = min(lo + bs, bat['n'])
        sub = _slice_batch(bat, lo, hi)
        handles = S.build_handles(spec, sources, lo, hi, pos)
        handles += [block(s).register_forward_hook(make_freeze_hook(frozen[s][lo:hi], FREEZE_POS))
                    for s in freeze]
        try:
            with torch.no_grad():
                lg = D.model(sub['ids_r']).logits[:, -1, :].float()
        finally:
            for h in handles:
                h.remove()
        r, i = posrec_and_iia(lg, sub)
        recs.append(r)
        iias.append(i)
        del lg
    return np.concatenate(recs), np.concatenate(iias)

## Evaluation batches and sources

Same pool, split and seed as `data_sharedrep.py`, so these numbers sit on the same 400 triples as
the saved conditions. One batch per case: `n1 larger` puts the clean maximum at the first number
slot, `n2 larger` at the second. Everything downstream is keyed by case.

The printed diagnostic is how far the clean and corrupted runs differ along each patched direction
— a patch can only act where its source moves.

In [ ]:
CASES = ['n1 larger', 'n2 larger']      # which number slot holds the clean maximum
SWEEP_LAYERS = [14, 15, 16]

if PLOT_ONLY:
    print('plot-only: skipping the per-case captures (they only feed the compute paths)')
else:
    pool = build_triples(N_EVAL + S.N_H14_DAS, SEED, low, high, gap)
    eval_triples = pool[:N_EVAL]
    assert np.array_equal(np.array(eval_triples, dtype=np.int64), R['eval_triples']), \
        'eval set must match the saved sharedrep run'

    SITES = [(k, l) for l in SWEEP_LAYERS for k in ('mlp', 'attn')]

    BAT, SRC, FRZ, DISP = {}, {}, {}, {}
    for case in CASES:
        b = make_batch(eval_triples, CASE_SLOT[case], T)
        b['pld_clean'], b['pld_corr'] = pos_anchors(b)
        BAT[case] = b
        SRC[case] = {LAYER: S.clean_hidden_at(b['ids_c'], LAYER),   # clean L13 residual
                     'ctx': S.capture_ctx(b['ids_c'], POS_B)}       # clean o_proj input @ n2, L14
        FRZ[case] = capture_corrupted(SITES, b['ids_r'], POS_B)

        # Patch-source displacement: |clean - corrupted| along each patched direction, at n2. Zero
        # means the patch writes back what is already there, i.e. a null intervention.
        ctx_r = S.capture_ctx(b['ids_r'], POS_B)
        h13_r = S.clean_hidden_at(b['ids_r'], LAYER)[:, POS_B, :].float()
        du = ((S.head_out(SRC[case]['ctx'], S.H_HEAD) - S.head_out(ctx_r, S.H_HEAD)) @ Q_h14).abs()
        dv = ((SRC[case][LAYER][:, POS_B, :].float() - h13_r) @ P_l13b).abs()
        DISP[case] = (float(du.mean()), float(dv.mean()))           # saved by the last cell
        print(f'{case:>10}: mean |clean - corr| along u {DISP[case][0]:7.3f}   '
              f'along v2 {DISP[case][1]:7.3f}')
        del ctx_r, h13_r, du, dv
        torch.cuda.empty_cache()

    print(f'captured corrupted output @ n2 for {len(SITES)} sites, {len(CASES)} cases: '
          + '  '.join(f'{k}{l}' for k, l in SITES))

## Sanity check

Freezing with no patch is the identity: the frozen value *is* what the corrupted run produces, so
position recovery must be 0 and IIA 0.

In [ ]:
if PLOT_ONLY:
    print('plot-only: sanity check skipped (its result is in the checks file)')
else:
    SANITY = {}
    for case in CASES:
        for fz in [(('mlp', 14),), (('attn', 14),), (('mlp', 14), ('mlp', 15))]:
            rec, iia = eval_spec({}, BAT[case], SRC[case], POS_B, freeze=fz, frozen=FRZ[case])
            tag = '+'.join(f'{k}{l}' for k, l in fz)
            SANITY[case, tag] = (float(rec.mean()), float(iia.mean()))   # saved by the last cell
            print(f'{case:>10}, no patch, freeze {tag:<16}: posrec {rec.mean():+.6f}   '
                  f'IIA {iia.mean():.3f}')

## Patch × freeze grid

Patch arms: H14 $\mathbf{u}$ · L13 $\mathbf{v}_2$ @ $n_2$ · both. Freeze sets: none, each of
MLP/attention at L14–L16, and MLP14+MLP15 together. Run for both cases.

Which arm carries the interchange flips with the case. In `n1 larger` the contested number is
copied into $n_2$ by H14, so $\mathbf{u}$ is the live patch; in `n2 larger` it already sits at
$n_2$, so $\mathbf{v}_2$ is, and $\mathbf{u}$ then carries the loser — the same number in both
runs — which makes it near-null. The freeze columns are what is being compared, not the arms.

Freezing **attn14** subsumes the H14 $\mathbf{u}$ patch — that patch is written into layer 14's
`o_proj` output, which the freeze overwrites — so those cells are a manipulation check, not an
independent result.

In [ ]:
FORCE_GRID = False        # True re-runs the patching even when results/ already holds it

# The arms carry the direction tensors, so they can only be built when the model is loaded; in
# plot-only mode the cell below fills ARMS with the names from the saved meta instead.
if not PLOT_ONLY:
    ARMS = {'H14 u':              dict(head=(S.H_HEAD, Q_h14)),
            'L13 v2 @n2':         dict(resid=[(LAYER, 'sub', P_l13b)]),
            'H14 u + L13 v2 @n2': dict(resid=[(LAYER, 'sub', P_l13b)], head=(S.H_HEAD, Q_h14))}

FREEZE_SETS = {'none': ()}
FREEZE_SETS.update({f'MLP{l}': (('mlp', l),) for l in SWEEP_LAYERS})
FREEZE_SETS.update({f'attn{l}': (('attn', l),) for l in SWEEP_LAYERS})
FREEZE_SETS['MLP14+15'] = (('mlp', 14), ('mlp', 15))

# The specs above are cheap, and the neuron cells need ARMS as real specs, so they are always
# rebuilt; only the patching is skipped. When cached, `RES` is left undefined here and the cell
# below loads it -- which is also where the tables are printed, so they appear either way.
if PLOT_ONLY or (os.path.exists(f'{RESULTS_DIR}/comparator_{REGIME}.npz') and not FORCE_GRID):
    print(f'results/comparator_{REGIME}.npz exists -- skipping the ~2 min of patching. '
          'Set FORCE_GRID = True to re-run it.')
else:
    t0 = time.time()
    RES = {}
    for case in CASES:
        for arm, spec in ARMS.items():
            for fz_name, fz in FREEZE_SETS.items():
                RES[case, arm, fz_name] = eval_spec(spec, BAT[case], SRC[case], POS_B,
                                                    freeze=fz, frozen=FRZ[case])
    print(f'{len(RES)} conditions in {time.time() - t0:.1f} s')

## Saved results

`results/comparator_2digit.npz` → `RES`, so the figures below can be redrawn in a fresh kernel
without the model and without re-running the patching. Skipped when the grid above has already run
in this kernel.

In [ ]:
NPZ_PATH  = f'{RESULTS_DIR}/comparator_{REGIME}.npz'
META_PATH = f'{RESULTS_DIR}/comparator_meta.json'


def key_of(case, arm, fz):
    """(case, patch arm, freeze set) -> the flat key these results are saved under."""
    return f'{case}__{arm}__{fz}'.replace(' ', '_').replace('+', 'and').replace('@', 'at')


# A kernel that has just run the grid keeps its own RES; `del RES` forces a reload from disk.
if 'RES' in globals():
    print(f'using the {len(RES)} conditions already in this kernel')
else:
    assert os.path.exists(NPZ_PATH), f'{NPZ_PATH} not found -- set FORCE_GRID = True above'
    Z, mt = dict(np.load(NPZ_PATH)), json.load(open(META_PATH))
    CASES, ARMS = mt['cases'], mt['arms']
    FREEZE_SETS = {k: tuple(tuple(s) for s in v) for k, v in mt['freeze_sets'].items()}
    SWEEP_LAYERS, N_EVAL, eval_triples = mt['sweep_layers'], mt['n_eval'], Z['eval_triples']
    RES = {(c, a, f): (Z[f'posrec_{key_of(c, a, f)}'], Z[f'iia_{key_of(c, a, f)}'])
           for c in CASES for a in ARMS for f in FREEZE_SETS}
    print(f'loaded {len(RES)} conditions from {NPZ_PATH}   cases {CASES}, n={N_EVAL}')

hdr = f'{"":<22}' + ''.join(f'{f:>12}' for f in FREEZE_SETS)
for metric, idx in (('position recovery', 0), ('IIA', 1)):
    for case in CASES:
        print(f'\n--- {metric}, {case}  (patched @ n2, freeze @ n2, n={N_EVAL})\n{hdr}')
        for arm in ARMS:
            row = ''.join(f'{RES[case, arm, f][idx].mean():12.3f}' for f in FREEZE_SETS)
            print(f'{arm:<22}{row}')

## Figure 1 — effect of each freeze on every patch arm

One column per case, rows shared on the y axis. Dashed line: that case's unfrozen joint patch.

In [ ]:
ARM_COL = {'H14 u': COL['h14'], 'L13 v2 @n2': COL['pc1'], 'H14 u + L13 v2 @n2': COL['h14']}
ARM_LABEL = {'H14 u': f'H14 {U}', 'L13 v2 @n2': f'L13 {V2} @ $n_2$',
             'H14 u + L13 v2 @n2': f'H14 {U} + L13 {V2} @ $n_2$'}
JOINT = 'H14 u + L13 v2 @n2'
METRICS = [('iia', 1, 'IIA', (0, 1.05)), ('posrec', 0, 'position recovery', None)]

fz_names = list(FREEZE_SETS)
x = np.arange(len(fz_names))
W = 0.8 / len(ARMS)

fig, axes = plt.subplots(len(METRICS), len(CASES), figsize=(8.6 * len(CASES), 8.4),
                         sharex='col', sharey='row', squeeze=False)
for ci, case in enumerate(CASES):
    for ri, (metric, idx, ylab, ylim) in enumerate(METRICS):
        ax = axes[ri, ci]
        for j, arm in enumerate(ARMS):
            m = np.array([RES[case, arm, f][idx].mean() for f in fz_names])
            se = np.array([RES[case, arm, f][idx].std() / np.sqrt(N_EVAL) for f in fz_names])
            ax.bar(x + (j - (len(ARMS) - 1) / 2) * W, m, W, yerr=se, capsize=2,
                   color=ARM_COL[arm], alpha=0.9, edgecolor='white', linewidth=0,
                   hatch=JOINT_HATCH if '+' in arm else '', label=ARM_LABEL[arm])
        ax.axhline(RES[case, JOINT, 'none'][idx].mean(), ls='--', color=COL['ceil'], lw=1.1)
        if ylim: ax.set_ylim(*ylim)
        style_axes(ax, ylabel=ylab if ci == 0 else None)
        if ri == 0:
            ax.set_title(case, fontsize=TITLE_FS)
        if ri == len(METRICS) - 1:
            ax.axhline(0.0, color='0.3', lw=0.8)
            ax.set_xticks(x)
            ax.set_xticklabels([f + ('*' if f == 'attn14' else '') for f in fz_names],
                               fontsize=TICK_FS)
            ax.set_xlabel('frozen sub-block (output @ $n_2$ pinned to the corrupted run)',
                          fontsize=LABEL_FS)
axes[0, 0].legend(fontsize=LEGEND_FS, frameon=False, ncol=3, loc='upper right')
plt.tight_layout(); savefig(fig, f'{REGIME}_freeze_grid'); plt.show()
print('* attn14 overwrites the H14 u patch itself -- manipulation check, not an independent result.')

## Figure 2 — L14–L16 sweep under the joint patch

MLP vs attention at each layer, one row per case, as a fraction of that case's unfrozen
joint-patch effect.

In [ ]:
REL = True        # bars as a fraction of the unfrozen joint value; False = raw units
REL_MIN = 0.05    # a near-zero reference would blow the ratio up: that panel stays in raw units

KIND_COL = {'mlp': COL['full'], 'attn': COL['das']}
KIND_LAB = {'mlp': 'MLP frozen', 'attn': 'attention frozen'}
xs = np.arange(len(SWEEP_LAYERS))

fig, axes = plt.subplots(len(CASES), 2, figsize=(13.5, 4.9 * len(CASES)), squeeze=False)
for ri, case in enumerate(CASES):
    for ci, (metric, idx, ylab) in enumerate([('iia', 1, 'IIA'), ('posrec', 0, 'position recovery')]):
        ax = axes[ri, ci]
        ref = RES[case, JOINT, 'none'][idx].mean()
        rel = REL and abs(ref) > REL_MIN
        sc = ref if rel else 1.0
        for j, kind in enumerate(('mlp', 'attn')):
            name = (lambda l, k=kind: f'MLP{l}' if k == 'mlp' else f'attn{l}')
            m = np.array([RES[case, JOINT, name(l)][idx].mean() for l in SWEEP_LAYERS])
            se = np.array([RES[case, JOINT, name(l)][idx].std() / np.sqrt(N_EVAL)
                           for l in SWEEP_LAYERS])
            b = ax.bar(xs + (j - 0.5) * 0.36, m / sc, 0.36, yerr=se / abs(sc), capsize=3,
                       color=KIND_COL[kind], alpha=0.9, edgecolor='white', linewidth=0,
                       label=KIND_LAB[kind])
            ax.bar_label(b, fmt='%.2f', fontsize=TICK_FS - 2, padding=3)
        ax.axhline(1.0 if rel else ref, ls='--', color=COL['ceil'], lw=1.1,
                   label='no freeze' if (ri == 0 and metric == 'iia') else None)
        ax.axhline(0.0, color='0.3', lw=0.8)
        ax.set_xticks(xs); ax.set_xticklabels([f'L{l}' for l in SWEEP_LAYERS], fontsize=TICK_FS)
        style_axes(ax, ylabel=(f'{ylab}  (/ unfrozen)' if rel else ylab),
                   title=f'{case}   (unfrozen {ylab} = {ref:.3f})')
axes[0, 0].legend(fontsize=LEGEND_FS, frameon=False, loc='lower left')
fig.suptitle(f'joint patch: H14 {U} + L13 {V2} @ $n_2$', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, f'{REGIME}_layer_sweep'); plt.show()

## Save

In [ ]:
arrays = {}
for (case, arm, fz), (rec, iia) in RES.items():
    arrays[f'posrec_{key_of(case, arm, fz)}'] = rec.astype(np.float32)
    arrays[f'iia_{key_of(case, arm, fz)}'] = iia.astype(np.float32)
arrays['eval_triples'] = np.array(eval_triples, dtype=np.int64)

meta = dict(regime=REGIME, model=D.MODELNAME, cases=CASES, seed=SEED, n_eval=N_EVAL,
            l13_layer=LAYER, h_layer=S.H_LAYER, h_head=S.H_HEAD, pos_second=int(POS_B),
            freeze_pos=int(FREEZE_POS), sweep_layers=SWEEP_LAYERS,
            arms=list(ARMS), freeze_sets={k: [list(s) for s in v] for k, v in FREEZE_SETS.items()},
            dirs_from=f'{RESULTS_DIR}/sharedrep_{REGIME}.npz',
            summary={key_of(c, a, f): dict(posrec=float(RES[c, a, f][0].mean()),
                                           iia=float(RES[c, a, f][1].mean()))
                     for c in CASES for a in ARMS for f in FREEZE_SETS})
np.savez_compressed(NPZ_PATH, **arrays)
with open(META_PATH, 'w') as fh:
    json.dump(meta, fh, indent=2, default=float)
print(f'{NPZ_PATH}: {os.path.getsize(NPZ_PATH) / 1e6:.3f} MB   {len(arrays)} arrays')

# Neuron level

Which neurons inside MLP14 / MLP15 carry the comparison, and what do they read from the
$\mathbf{v}_1,\mathbf{v}_2$ plane. A neuron is the post-SwiGLU scalar feeding `down_proj`
(`e11/which_neurons_compare.ipynb`'s definition), $d_{ff}=18944$ per layer.

1. **Rank** — attribution patching: $s_i\approx(a_i^{\mathrm{corr}}-a_i^{\mathrm{patched}})\,
   \partial\,\mathrm{PLD}/\partial a_i$, one backward pass for all neurons.
2. **Verify** — freeze the top $k$, against random-$k$ and bottom-$k$ controls.
3. **Inject** — the sufficiency direction: the *clean* MLP output at $n_2$, no upstream patch.
4. **Tune** — top neurons' activation over span($\mathbf{v}_1,\mathbf{v}_2$), and their
   $\gamma$-folded read-in weights in that plane.

Every block caches to `results/`; a block whose file exists is not recomputed. First run ~5 min,
later runs seconds. Set `FORCE = True` to recompute.

In [ ]:
FORCE = False          # True recomputes every cached block below

ATP_BS    = 16         # forward+backward minibatch: the backward holds activations for L14+
ATP_SCALE = 100.0      # loss scaling, guards fp16 gradient underflow
NEURON_LAYERS = [14, 15]
K_GRID     = [1, 3, 10, 30, 100, 300, 1000, 3000]
RAND_SEEDS = [0, 1]
INJ_LAYERS = [[14], [14, 15], [14, 15, 16]]     # cumulative, so L15/L16 cannot undo L14 alone
N_TUNE_CLOUD = 1500    # cloud pairs for the tuning maps
N_TUNE_TOP   = 24      # neurons whose cloud activations are captured
N_ALIGN_TOP  = 100     # neurons scored for read-in alignment
N_ALIGN_BG   = 500     # random neurons, for the alignment null

ATP_PATH  = f'{RESULTS_DIR}/comparator_atp_{REGIME}.npz'
TOPK_PATH = f'{RESULTS_DIR}/comparator_topk_{REGIME}.npz'
INJ_PATH  = f'{RESULTS_DIR}/comparator_inject_{REGIME}.npz'
TUNE_PATH = f'{RESULTS_DIR}/comparator_tune_{REGIME}.npz'

D_FF = D.model.config.intermediate_size if D.model is not None else None
ck = (lambda s: s.replace(' ', '_'))            # case name -> array-key fragment


def cached_npz(path, compute):
    """Load `path` if it is on disk, else run `compute()` -> {name: array}, save it, return it.

    This is the do-not-run-it-again guard: each block below is skipped once its file exists.
    Delete the file or set FORCE = True to recompute.
    """
    if os.path.exists(path) and not FORCE:
        d = dict(np.load(path))
        print(f'loaded {path}   ({len(d)} arrays)')
        return d
    assert 'BAT' in globals(), f'{path} is not cached yet -- run the cells above (model needed)'
    t = time.time()
    d = compute()
    np.savez_compressed(path, **d)
    print(f'computed and saved {path}   ({len(d)} arrays, {time.time() - t:.1f} s)')
    return d


def mlp_of(layer):
    return D.model.model.layers[layer].mlp


def capture_neurons(ids, layers, pos, bs=EVAL_BS, idx=None, patch=None):
    """{layer: (n, d_ff)} post-SwiGLU activations at `pos` -- the `down_proj` input.

    `idx` (a {layer: LongTensor}) keeps only those neurons. `patch(lo, hi)` optionally registers
    interventions per minibatch, so the same routine reads the corrupted run and the patched one.
    """
    store, chunks = {}, {l: [] for l in layers}

    def mk(l):
        def _pre(mod, inp):
            a = inp[0][:, pos, :].detach().float()
            store[l] = a if idx is None else a[:, idx[l]]
        return _pre

    hs = [mlp_of(l).down_proj.register_forward_pre_hook(mk(l)) for l in layers]
    try:
        for lo in range(0, ids.shape[0], bs):
            hi = min(lo + bs, ids.shape[0])
            extra = patch(lo, hi) if patch is not None else []
            try:
                with torch.no_grad():
                    _ = D.model(ids[lo:hi])
            finally:
                for h in extra:
                    h.remove()
            for l in layers:
                chunks[l].append(store.pop(l))
    finally:
        for h in hs:
            h.remove()
    return {l: torch.cat(v, dim=0) for l, v in chunks.items()}


def make_neuron_freeze_hook(idx, vals, pos):
    """`down_proj` pre-hook pinning neurons `idx` at `pos` to `vals` (B, len(idx)).

    Written in place: this tensor is `act_fn(gate(x)) * up(x)`, freshly allocated inside the MLP's
    forward and consumed only by `down_proj`, so nothing else observes the mutation. (The
    whole-module freeze has to clone, because it rewrites an output the residual add also reads.)
    """
    def _pre(mod, inp, _i=idx, _v=vals, _p=pos):
        inp[0][:, _p, _i] = _v.to(inp[0].dtype)
    return _pre


print(f'd_ff = {D_FF}   neuron layers {NEURON_LAYERS}   '
      f'{"" if D_FF is None else 2 * D_FF} candidates pooled')

## Rank — attribution patching

Base run = corrupted + the joint $\mathbf{v}_1,\mathbf{v}_2$ patch, i.e. the regime the freeze
happens in; source = the corrupted activations. PLD $=$ logit[`tok_nc`] $-$ logit[`tok_b`], the
numerator of position recovery (IIA is an argmax and has no gradient). Ranked on the 128 held-out
triples, never on the 400 the curve is reported on.

Large **negative** score = freezing that neuron destroys the effect.

In [ ]:
def _atp():
    rank_triples = pool[N_EVAL:]                      # the DAS split, disjoint from eval_triples
    assert not set(map(tuple, rank_triples)) & set(map(tuple, eval_triples))
    out = {'rank_triples': np.array(rank_triples, dtype=np.int64)}

    for case in CASES:
        rbat = make_batch(rank_triples, CASE_SLOT[case], T)
        rsrc = {LAYER: S.clean_hidden_at(rbat['ids_c'], LAYER),
                'ctx': S.capture_ctx(rbat['ids_c'], POS_B)}
        a_corr = capture_neurons(rbat['ids_r'], NEURON_LAYERS, POS_B, bs=ATP_BS)

        attr = {l: np.zeros(D_FF, dtype=np.float64) for l in NEURON_LAYERS}
        hs_store, a_store = {}, {}

        def mk_atp(l):
            def _pre(mod, inp):
                x = inp[0]
                # A zero handle that requires grad: the model's own parameters are frozen, so this
                # is what creates a graph from this activation down to the logits.
                h = torch.zeros(x.shape[0], x.shape[2], device=x.device, dtype=torch.float32,
                                requires_grad=True)
                hs_store[l], a_store[l] = h, x[:, POS_B, :].detach().float()
                xn = x.clone()
                xn[:, POS_B, :] = xn[:, POS_B, :] + h.to(x.dtype)
                return (xn,) + inp[1:]
            return _pre

        for lo in range(0, rbat['n'], ATP_BS):
            hi = min(lo + ATP_BS, rbat['n'])
            sub = _slice_batch(rbat, lo, hi)
            handles = S.build_handles(ARMS[JOINT], rsrc, lo, hi, POS_B)
            handles += [mlp_of(l).down_proj.register_forward_pre_hook(mk_atp(l))
                        for l in NEURON_LAYERS]
            try:
                with torch.enable_grad():
                    lg = D.model(sub['ids_r']).logits[:, -1, :].float()
                    ix = torch.arange(sub['n'], device=device)
                    # sum, not mean: each example's PLD depends only on its own handle row, so the
                    # gradient is then the per-example dPLD/da with no 1/B to undo.
                    (ATP_SCALE * (lg[ix, sub['tok_nc']] - lg[ix, sub['tok_b']]).sum()).backward()
            finally:
                for h in handles:
                    h.remove()
            for l in NEURON_LAYERS:
                g = (hs_store[l].grad / ATP_SCALE)
                attr[l] += ((a_corr[l][lo:hi] - a_store[l]) * g).sum(0).double().cpu().numpy()
            hs_store.clear(); a_store.clear()
            del lg
            torch.cuda.empty_cache()

        for l in NEURON_LAYERS:
            out[f'attr_{ck(case)}_L{l}'] = (attr[l] / rbat['n']).astype(np.float32)
        print(f'{case:>10}: attribution done, '
              + '  '.join(f'L{l} min {attr[l].min() / rbat["n"]:+.4f}' for l in NEURON_LAYERS),
              flush=True)
    return out


ATP = cached_npz(ATP_PATH, _atp)

# Pool both layers into one ranking; ascending = most damaging to freeze first.
POOL_LAYER = np.concatenate([np.full(len(ATP[f'attr_{ck(CASES[0])}_L{l}']), l)
                             for l in NEURON_LAYERS])
POOL_WITHIN = np.concatenate([np.arange(len(ATP[f'attr_{ck(CASES[0])}_L{l}']))
                              for l in NEURON_LAYERS])
SCORES = {c: np.concatenate([ATP[f'attr_{ck(c)}_L{l}'] for l in NEURON_LAYERS]) for c in CASES}
ORDER = {c: np.argsort(SCORES[c]) for c in CASES}

for c in CASES:
    top = ORDER[c][:100]
    print(f'{c:>10}: top-100 split L14/L15 = {(POOL_LAYER[top] == 14).sum()}/'
          f'{(POOL_LAYER[top] == 15).sum()}   score[0] {SCORES[c][ORDER[c][0]]:+.4f}   '
          f'score[99] {SCORES[c][top[-1]]:+.4f}')
ov = len(set(ORDER[CASES[0]][:100]) & set(ORDER[CASES[1]][:100]))
print(f'top-100 overlap between cases: {ov}/100  (chance {100 * 100 / len(POOL_LAYER):.2f})')

## Verify — freeze the top $k$

Top-$k$ against random-$k$ (2 seeds) and bottom-$k$, both cases, on the 400 eval triples with the
joint patch. The $k=2d_{ff}$ endpoint freezes every neuron in both layers, which *is* the
MLP14+15 module freeze (`down_proj` has no bias, so it is exactly linear) — printed as a check
against the value already measured.

In [ ]:
def _topk():
    out = {}
    NEUR_CORR = {c: capture_neurons(BAT[c]['ids_r'], NEURON_LAYERS, POS_B) for c in CASES}

    def sel_of(case, idx_pool):
        """pooled indices -> {layer: LongTensor of that layer's neuron indices}."""
        return {l: torch.as_tensor(POOL_WITHIN[idx_pool[POOL_LAYER[idx_pool] == l]],
                                   device=device, dtype=torch.long) for l in NEURON_LAYERS}

    def run(case, sel, bs=EVAL_BS):
        bat, recs, iias = BAT[case], [], []
        for lo in range(0, bat['n'], bs):
            hi = min(lo + bs, bat['n'])
            sub = _slice_batch(bat, lo, hi)
            handles = S.build_handles(ARMS[JOINT], SRC[case], lo, hi, POS_B)
            for l, ii in sel.items():
                if len(ii):
                    handles.append(mlp_of(l).down_proj.register_forward_pre_hook(
                        make_neuron_freeze_hook(ii, NEUR_CORR[case][l][lo:hi][:, ii], POS_B)))
            try:
                with torch.no_grad():
                    lg = D.model(sub['ids_r']).logits[:, -1, :].float()
            finally:
                for h in handles:
                    h.remove()
            r, i = posrec_and_iia(lg, sub)
            recs.append(r); iias.append(i)
            del lg
        return np.concatenate(recs), np.concatenate(iias)

    n_pool = len(POOL_LAYER)
    for case in CASES:
        for k in K_GRID:
            series = {'top': ORDER[case][:k], 'bottom': ORDER[case][-k:]}
            for s in RAND_SEEDS:
                series[f'rand{s}'] = np.random.default_rng(s).choice(n_pool, k, replace=False)
            for name, idx_pool in series.items():
                rec, iia = run(case, sel_of(case, idx_pool))
                out[f'posrec_{ck(case)}_{name}_k{k}'] = rec.astype(np.float32)
                out[f'iia_{ck(case)}_{name}_k{k}'] = iia.astype(np.float32)
        rec, iia = run(case, sel_of(case, np.arange(n_pool)))        # every neuron
        out[f'posrec_{ck(case)}_all'] = rec.astype(np.float32)
        out[f'iia_{ck(case)}_all'] = iia.astype(np.float32)
        print(f'{case:>10}: all-neuron endpoint  posrec {rec.mean():.3f}  IIA {iia.mean():.3f}   '
              f'module MLP14+15 freeze was  posrec {RES[case, JOINT, "MLP14+15"][0].mean():.3f}  '
              f'IIA {RES[case, JOINT, "MLP14+15"][1].mean():.3f}', flush=True)
    return out


TOPK = cached_npz(TOPK_PATH, _topk)

SERIES = ['top', 'bottom'] + [f'rand{s}' for s in RAND_SEEDS]
for case in CASES:
    ref = RES[case, JOINT, 'none'][1].mean()
    print(f'\n--- IIA vs k, {case}   (no freeze {ref:.3f}, all neurons '
          f'{TOPK[f"iia_{ck(case)}_all"].mean():.3f})')
    print(f'{"":>8}' + ''.join(f'{k:>9}' for k in K_GRID))
    for name in SERIES:
        print(f'{name:>8}' + ''.join(f'{TOPK[f"iia_{ck(case)}_{name}_k{k}"].mean():9.3f}'
                                     for k in K_GRID))

In [ ]:
SER_COL = {'top': COL['full'], 'bottom': COL['pc1'], 'rand0': COL['ceil'], 'rand1': COL['ceil']}
SER_LAB = {'top': 'top-$k$ by attribution', 'bottom': 'bottom-$k$',
           'rand0': 'random $k$', 'rand1': '_nolegend_'}

fig, axes = plt.subplots(len(CASES), 2, figsize=(13.5, 4.9 * len(CASES)), squeeze=False)
for ri, case in enumerate(CASES):
    for ci, (idx, ylab) in enumerate([(1, 'IIA'), (0, 'position recovery')]):
        ax = axes[ri, ci]
        ref = RES[case, JOINT, 'none'][idx].mean()
        for name in SERIES:
            m = np.array([TOPK[f'{["posrec", "iia"][idx]}_{ck(case)}_{name}_k{k}'].mean()
                          for k in K_GRID])
            se = np.array([TOPK[f'{["posrec", "iia"][idx]}_{ck(case)}_{name}_k{k}'].std()
                           / np.sqrt(N_EVAL) for k in K_GRID])
            ax.errorbar(K_GRID, m, yerr=se, marker='o', ms=5, lw=2 if name == 'top' else 1.4,
                        color=SER_COL[name], ls='--' if name.startswith('rand') else '-',
                        capsize=2, label=SER_LAB[name])
        ax.axhline(ref, ls='--', color='0.4', lw=1.1, label='no freeze' if ci == 0 else None)
        ax.axhline(TOPK[f'{["posrec", "iia"][idx]}_{ck(case)}_all'].mean(), ls=':', color='0.4',
                   lw=1.1, label='all neurons' if ci == 0 else None)
        ax.set_xscale('log')
        style_axes(ax, xlabel='$k$ neurons frozen at $n_2$ (pooled over L14, L15)',
                   ylabel=ylab, title=case)
axes[0, 0].legend(fontsize=LEGEND_FS - 1, frameon=False, loc='lower left')
plt.tight_layout(); savefig(fig, f'{REGIME}_topk_neurons'); plt.show()

## Inject — the sufficiency direction

The *clean* MLP output at $n_2$ written into an otherwise untouched corrupted run (no upstream
patch). Same hook as the freeze, different source.

Two read-outs, because they say different things: `IIA (tok_nc)` is the model naming the winning
**slot**, `IIA (tok_a)` is it naming the clean run's winning **value**. The freeze cannot separate
"MLP writes the comparison outcome" from "MLP writes the answer".

In [ ]:
def _inject():
    out = {}
    for case in CASES:
        bat = BAT[case]
        # The clean run's winning number's first token: make_batch keeps tok_nc / tok_b, which are
        # both corrupted-run tokens. Leading digits are distinct inside a triple, so this is
        # unambiguous.
        strs = [D.prompt_gen(list(p)) for p in bat['clean_pairs']]
        enc = tokenizer(strs, return_tensors='pt', return_offsets_mapping=True)
        tok_a = torch.tensor(
            [int(enc['input_ids'][i, D.num_tok_span(strs[i], list(bat['clean_pairs'][i]),
                                                    bat['win_slot'],
                                                    enc['offset_mapping'][i].numpy())[0]])
             for i in range(bat['n'])], device=device)
        out[f'tok_a_{ck(case)}'] = tok_a.cpu().numpy()
        # capture_corrupted is generic -- given the CLEAN ids it returns the clean MLP outputs.
        clean_mlp = capture_corrupted([('mlp', l) for l in INJ_LAYERS[-1]], bat['ids_c'], POS_B)

        for layers in [[]] + INJ_LAYERS:              # [] = the untouched corrupted baseline
            recs, i_nc, i_a = [], [], []
            for lo in range(0, bat['n'], EVAL_BS):
                hi = min(lo + EVAL_BS, bat['n'])
                sub = _slice_batch(bat, lo, hi)
                handles = [block(('mlp', l)).register_forward_hook(
                    make_freeze_hook(clean_mlp[('mlp', l)][lo:hi], POS_B)) for l in layers]
                try:
                    with torch.no_grad():
                        lg = D.model(sub['ids_r']).logits[:, -1, :].float()
                finally:
                    for h in handles:
                        h.remove()
                r, i = posrec_and_iia(lg, sub)
                recs.append(r); i_nc.append(i)
                i_a.append((lg.argmax(-1) == tok_a[lo:hi]).float().cpu().numpy())
                del lg
            tag = 'none' if not layers else '+'.join(f'MLP{l}' for l in layers)
            out[f'posrec_{ck(case)}_{tag}'] = np.concatenate(recs).astype(np.float32)
            out[f'iia_nc_{ck(case)}_{tag}'] = np.concatenate(i_nc).astype(np.float32)
            out[f'iia_a_{ck(case)}_{tag}'] = np.concatenate(i_a).astype(np.float32)
    return out


INJ = cached_npz(INJ_PATH, _inject)

INJ_TAGS = ['none'] + ['+'.join(f'MLP{l}' for l in ls) for ls in INJ_LAYERS]
for case in CASES:
    print(f'\n--- inject clean MLP output @ n2, {case}  (no upstream patch, n={N_EVAL})')
    print(f'{"injected":>18}{"posrec":>10}{"IIA (tok_nc)":>15}{"IIA (tok_a)":>14}')
    for tag in INJ_TAGS:
        print(f'{tag:>18}{INJ[f"posrec_{ck(case)}_{tag}"].mean():10.3f}'
              f'{INJ[f"iia_nc_{ck(case)}_{tag}"].mean():15.3f}'
              f'{INJ[f"iia_a_{ck(case)}_{tag}"].mean():14.3f}')

In [ ]:
MET = [('iia_nc', 'IIA (names the winning slot, tok_nc)'),
       ('iia_a', 'IIA (names the clean winner, tok_a)'),
       ('posrec', 'position recovery')]
xi = np.arange(len(INJ_TAGS))
Wb = 0.8 / len(CASES)

fig, axes = plt.subplots(1, len(MET), figsize=(5.4 * len(MET), 5.0))
for ax, (key, ylab) in zip(axes, MET):
    for j, case in enumerate(CASES):
        m = np.array([INJ[f'{key}_{ck(case)}_{t}'].mean() for t in INJ_TAGS])
        se = np.array([INJ[f'{key}_{ck(case)}_{t}'].std() / np.sqrt(N_EVAL) for t in INJ_TAGS])
        ax.bar(xi + (j - (len(CASES) - 1) / 2) * Wb, m, Wb, yerr=se, capsize=3,
               color=[COL['h14'], COL['pc1']][j], alpha=0.9, edgecolor='white', linewidth=0,
               label=case)
    ax.axhline(0.0, color='0.3', lw=0.8)
    ax.set_xticks(xi); ax.set_xticklabels(INJ_TAGS, rotation=20, ha='right', fontsize=TICK_FS)
    style_axes(ax, ylabel=ylab)
axes[0].legend(fontsize=LEGEND_FS, frameon=False)
fig.suptitle('clean MLP output injected at $n_2$, no upstream patch', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, f'{REGIME}_inject_mlp'); plt.show()

## Tune — the top neurons in the $\mathbf{v}_1,\mathbf{v}_2$ plane

Plane basis: $\mathbf{e}_1=\mathbf{v}_1$, $\mathbf{e}_2$ = $\mathbf{v}_2$ orthogonalised against it
(`makefig_sharedreps.ipynb` Fig 6's convention), read at the pre-MLP L14 residual — the input to
`post_attention_layernorm`, the site sharedreps found the rank-2 patch works at.
Coordinates are centred on the cloud mean and signed so each rises with its own number.

Read-in side: a SwiGLU neuron has two, $a_i=\mathrm{silu}(w_{g,i}\!\cdot\!\tilde x)\,
(w_{u,i}\!\cdot\!\tilde x)$ with $\tilde x=\mathrm{RMSNorm}(x)\odot\gamma$, so both rows are folded
with $\gamma$ before any cosine. The gate is the switch, so a threshold unit's prediction is on
$w_g$. Chance in-plane fraction for a random vector is $\sqrt{2/d}$.

In [ ]:
def _tune():
    v1 = torch.tensor(h14_das_np.astype(np.float32), device=device)
    v2 = torch.tensor(l13_pc1b_np.astype(np.float32), device=device)
    e1 = v1 / v1.norm()
    e2 = v2 - (v2 @ e1) * e1
    e2 = e2 / e2.norm()

    top = ORDER[CASES[0]][:N_TUNE_TOP]              # ranked in the case where both arms are live
    idx = {l: torch.as_tensor(POOL_WITHIN[top[POOL_LAYER[top] == l]], device=device,
                              dtype=torch.long) for l in NEURON_LAYERS}

    pairs = np.array(S.sample_cloud_pairs(N_TUNE_CLOUD, SEED + 11, low, high), dtype=np.int64)
    coords, acts = [], {l: [] for l in NEURON_LAYERS}
    store = {}
    # The pre-MLP L14 residual = the input to post_attention_layernorm. (The MLP module's own
    # input is that vector already normalised and scaled by gamma, which is a different frame.)
    hs = [D.model.model.layers[14].post_attention_layernorm.register_forward_pre_hook(
              lambda m, i: store.__setitem__('x', i[0][:, POS_B, :].detach().float()))]
    hs += [mlp_of(l).down_proj.register_forward_pre_hook(
               (lambda _l: (lambda m, i: store.__setitem__(_l, i[0][:, POS_B, idx[_l]]
                                                           .detach().float())))(l))
           for l in NEURON_LAYERS]
    try:
        for i0 in range(0, len(pairs), EVAL_BS):
            chunk = pairs[i0:i0 + EVAL_BS]
            ids = tokenizer([D.prompt_gen(list(p)) for p in chunk],
                            return_tensors='pt')['input_ids'].to(device)
            with torch.no_grad():
                _ = D.model(ids)
            x = store.pop('x')
            coords.append(torch.stack([x @ e1, x @ e2], dim=1).cpu().numpy())
            for l in NEURON_LAYERS:
                acts[l].append(store.pop(l).cpu().numpy())
    finally:
        for h in hs:
            h.remove()

    pq = np.concatenate(coords).astype(np.float64)
    y1, y2 = pairs[:, 0].astype(np.float64), pairs[:, 1].astype(np.float64)
    pq = pq - pq.mean(0)
    sgn = np.sign([np.corrcoef(pq[:, 0], np.log(y1))[0, 1], np.corrcoef(pq[:, 1], np.log(y2))[0, 1]])
    pq = pq * sgn                                    # each axis rises with its own number

    # The difference axis measured on this cloud: the in-plane direction along which log y1 - log y2
    # grows fastest. A comparator neuron's read-in should lie along it, not along e1 or e2.
    w = np.linalg.lstsq(np.c_[pq, np.ones(len(pq))], np.log(y1) - np.log(y2), rcond=None)[0][:2]
    w = w / np.linalg.norm(w)

    out = {'pq': pq.astype(np.float32), 'y1': y1, 'y2': y2, 'diff_axis': w.astype(np.float32),
           'sgn': sgn.astype(np.float32), 'top_pooled': top.astype(np.int64),
           'top_layer': POOL_LAYER[top].astype(np.int64),
           'top_within': POOL_WITHIN[top].astype(np.int64)}
    for l in NEURON_LAYERS:
        out[f'acts_L{l}'] = np.concatenate(acts[l]).astype(np.float32)
        out[f'idx_L{l}'] = idx[l].cpu().numpy()

    # ---- read-in alignment, weight space only (no forward pass) ------------------------------
    E = torch.stack([e1 * float(sgn[0]), e2 * float(sgn[1])], dim=1)      # (d_model, 2)
    rng = np.random.default_rng(0)
    for case in CASES:
        sets = {'top': ORDER[case][:N_ALIGN_TOP],
                'bg': rng.choice(len(POOL_LAYER), N_ALIGN_BG, replace=False)}
        for nm, pooled in sets.items():
            for which in ('gate', 'up'):
                frac, ang, lay = [], [], []
                for l in NEURON_LAYERS:
                    ii = POOL_WITHIN[pooled[POOL_LAYER[pooled] == l]]
                    if not len(ii):
                        continue
                    g = D.model.model.layers[l].post_attention_layernorm.weight.detach().float()
                    mat = mlp_of(l).gate_proj if which == 'gate' else mlp_of(l).up_proj
                    W = mat.weight.detach()[torch.as_tensor(ii, device=device)].float() * g
                    P = (W @ E).cpu().numpy()                    # (n, 2), components on e1 / e2
                    frac.append(np.linalg.norm(P, axis=1) / W.norm(dim=1).cpu().numpy())
                    ang.append(np.degrees(np.arctan2(P[:, 1], P[:, 0])))
                    lay.append(np.full(len(ii), l))
                out[f'align_frac_{ck(case)}_{nm}_{which}'] = np.concatenate(frac).astype(np.float32)
                out[f'align_ang_{ck(case)}_{nm}_{which}'] = np.concatenate(ang).astype(np.float32)
                out[f'align_lay_{ck(case)}_{nm}_{which}'] = np.concatenate(lay).astype(np.int64)
    return out


TUNE = cached_npz(TUNE_PATH, _tune)
DIFF_ANG = float(np.degrees(np.arctan2(TUNE['diff_axis'][1], TUNE['diff_axis'][0])))
print(f"plane coords {TUNE['pq'].shape}   difference axis in-plane at {DIFF_ANG:+.1f} deg "
      f"(e1 = v1 at 0 deg, e2 = v2-perp at 90 deg)")
print(f"chance in-plane fraction sqrt(2/d) = {np.sqrt(2 / M['d_model']):.4f}")

In [ ]:
N_SHOW = 6            # top neurons drawn as tuning maps

pq, y1v, y2v = TUNE['pq'], TUNE['y1'], TUNE['y2']
show = list(range(min(N_SHOW, len(TUNE['top_pooled']))))

fig, axes = plt.subplots(2, len(show), figsize=(3.1 * len(show), 6.6), squeeze=False)
for c, r in enumerate(show):
    l, wi = int(TUNE['top_layer'][r]), int(TUNE['top_within'][r])
    col = int(np.where(TUNE[f'idx_L{l}'] == wi)[0][0])
    a = TUNE[f'acts_L{l}'][:, col]
    for row, (cvals, lab) in enumerate([(a, f'L{l} n{wi}  (rank {r + 1})'),
                                        (np.sign(np.log(y1v) - np.log(y2v)), '$y_1>y_2$')]):
        ax = axes[row, c]
        sc = ax.scatter(pq[:, 0], pq[:, 1], c=cvals, s=5, alpha=0.8, edgecolors='none',
                        cmap='RdBu_r' if row else 'viridis')
        t = np.linspace(-1, 1, 2) * np.abs(pq).max()
        # the boundary implied by the measured difference axis, through the cloud centre
        ax.plot(-TUNE['diff_axis'][1] * t, TUNE['diff_axis'][0] * t, '--', color='k', lw=1.1)
        ax.set_title(lab, fontsize=TICK_FS)
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(['neuron activation', 'sign of $\\log y_1-\\log y_2$'][row],
                          fontsize=TICK_FS)
        if row == 0:
            fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=8)
fig.suptitle(r'top neurons over span($\mathbf{v}_1,\mathbf{v}_2$) at MLP14 input  '
             '(dashed = boundary normal to the measured difference axis)', fontsize=TITLE_FS - 3)
plt.tight_layout(); savefig(fig, f'{REGIME}_neuron_tuning'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(CASES), figsize=(7.4 * len(CASES), 5.2), squeeze=False)
chance = np.sqrt(2 / M['d_model'])
for ci, case in enumerate(CASES):
    ax = axes[0, ci]
    bg = np.concatenate([TUNE[f'align_frac_{ck(case)}_bg_{w}'] for w in ('gate', 'up')])
    ax.axhspan(0, np.quantile(bg, 0.95), color='0.85', zorder=0,
               label=f'random neurons, 95th pct ({np.quantile(bg, 0.95):.3f})')
    for which, mk, colr in (('gate', 'o', COL['full']), ('up', '^', COL['das'])):
        ax.scatter(TUNE[f'align_ang_{ck(case)}_top_{which}'],
                   TUNE[f'align_frac_{ck(case)}_top_{which}'],
                   marker=mk, s=34, alpha=0.8, edgecolors='none', color=colr, label=which)
    for a0 in (DIFF_ANG, DIFF_ANG - 180 if DIFF_ANG > 0 else DIFF_ANG + 180):
        ax.axvline(a0, ls='--', color='k', lw=1.1)
    ax.axhline(chance, ls=':', color='0.4', lw=1.0)
    ax.set_xlim(-180, 180); ax.set_xticks(range(-180, 181, 90))
    style_axes(ax, xlabel=r'read-in angle in the plane (deg; 0 = $\mathbf{v}_1$)',
               ylabel='in-plane fraction of the read-in weight', title=case)
axes[0, 0].legend(fontsize=LEGEND_FS - 2, frameon=False, loc='upper left')
fig.suptitle(f'top-{N_ALIGN_TOP} neurons by attribution; dashed = measured difference axis '
             f'({DIFF_ANG:+.0f} deg)', fontsize=TITLE_FS - 2)
plt.tight_layout(); savefig(fig, f'{REGIME}_neuron_alignment'); plt.show()

## Top-10 neurons — receptive fields and read directions

Ranked by attribution in the `n1 larger` case (the ranking the tuning maps above use). Receptive
fields are a dense $(y_1,y_2)$ grid of prompts, not the natural cloud, so every cell is measured
rather than interpolated. Read directions are the $\gamma$-folded `gate_proj` / `up_proj` rows
projected into span($\mathbf{v}_1,\mathbf{v}_2$).

In [ ]:
RF_TOP    = 10                          # neurons to profile
RF_STEP   = 4                           # grid stride over each number axis
RF_CASE   = CASES[0]                    # ranking used; CASES[1] for the other case
RF_PATH   = f'{RESULTS_DIR}/comparator_rf_{REGIME}.npz'

rf_pooled = ORDER[RF_CASE][:RF_TOP]
rf_layer, rf_within = POOL_LAYER[rf_pooled], POOL_WITHIN[rf_pooled]


def _rf():
    grid = np.arange(low + 1, high, RF_STEP)                  # 11, 15, ... 99
    pairs = np.array([(a, b) for a in grid for b in grid], dtype=np.int64)
    idx = {l: torch.as_tensor(rf_within[rf_layer == l], device=device, dtype=torch.long)
           for l in NEURON_LAYERS}

    ids = []
    for i0 in range(0, len(pairs), EVAL_BS):
        enc = tokenizer([D.prompt_gen(list(p)) for p in pairs[i0:i0 + EVAL_BS]],
                        return_tensors='pt')['input_ids']
        assert enc.shape[1] == T, 'every grid prompt must share the regime token layout'
        ids.append(enc.to(device))
    acts = capture_neurons(torch.cat(ids, dim=0), NEURON_LAYERS, POS_B, idx=idx)

    out = {'grid': grid.astype(np.int64), 'pairs': pairs,
           'rf_pooled': rf_pooled.astype(np.int64), 'rf_layer': rf_layer.astype(np.int64),
           'rf_within': rf_within.astype(np.int64),
           'rf_score': SCORES[RF_CASE][rf_pooled].astype(np.float32)}
    for l in NEURON_LAYERS:
        out[f'rf_acts_L{l}'] = acts[l].cpu().numpy().astype(np.float32)
        out[f'rf_idx_L{l}'] = idx[l].cpu().numpy()

    # Read-in directions in the plane, same signed basis as the tuning cell.
    v1 = torch.tensor(h14_das_np.astype(np.float32), device=device)
    v2 = torch.tensor(l13_pc1b_np.astype(np.float32), device=device)
    e1 = v1 / v1.norm()
    e2 = v2 - (v2 @ e1) * e1
    e2 = e2 / e2.norm()
    sg = TUNE['sgn']
    E = torch.stack([e1 * float(sg[0]), e2 * float(sg[1])], dim=1)
    for which in ('gate', 'up'):
        comp, frac = [], []
        for l, i in zip(rf_layer, rf_within):
            g = D.model.model.layers[int(l)].post_attention_layernorm.weight.detach().float()
            mat = mlp_of(int(l)).gate_proj if which == 'gate' else mlp_of(int(l)).up_proj
            w = mat.weight.detach()[int(i)].float() * g
            p = (w @ E).cpu().numpy()
            comp.append(p)
            frac.append(float(np.linalg.norm(p) / w.norm().cpu().numpy()))
        out[f'read_{which}'] = np.array(comp, dtype=np.float32)       # (RF_TOP, 2)
        out[f'readfrac_{which}'] = np.array(frac, dtype=np.float32)
    return out


RF = cached_npz(RF_PATH, _rf)

print(f'\ntop-{RF_TOP} neurons by attribution, {RF_CASE}   '
      f'(searched layers {NEURON_LAYERS}; L16 is not in the pool)')
print(f'{"rank":>5}{"layer":>7}{"neuron":>9}{"attr":>10}{"|read| gate":>13}{"|read| up":>11}'
      f'{"angle gate":>12}{"angle up":>10}')
for r in range(len(RF['rf_pooled'])):
    ga, ua = (np.degrees(np.arctan2(RF[f'read_{w}'][r, 1], RF[f'read_{w}'][r, 0]))
              for w in ('gate', 'up'))
    print(f'{r + 1:>5}{"L" + str(RF["rf_layer"][r]):>7}{RF["rf_within"][r]:>9}'
          f'{RF["rf_score"][r]:>10.4f}{RF["readfrac_gate"][r]:>13.4f}'
          f'{RF["readfrac_up"][r]:>11.4f}{ga:>11.0f}d{ua:>9.0f}d')
n14 = int((RF['rf_layer'] == 14).sum())
print(f'\nsplit: {n14} in L14, {len(RF["rf_layer"]) - n14} in L15   '
      f'(difference axis at {DIFF_ANG:+.0f} deg, chance |read| {np.sqrt(2 / M["d_model"]):.4f})')

In [ ]:
NCOL = 5

grid = RF['grid']
ext = [grid.min(), grid.max(), grid.min(), grid.max()]
nrow = int(np.ceil(len(RF['rf_pooled']) / NCOL))

fig, axes = plt.subplots(nrow, NCOL, figsize=(3.3 * NCOL, 3.35 * nrow), squeeze=False)
for r in range(nrow * NCOL):
    ax = axes[r // NCOL, r % NCOL]
    if r >= len(RF['rf_pooled']):
        ax.axis('off')
        continue
    l, wi = int(RF['rf_layer'][r]), int(RF['rf_within'][r])
    col = int(np.where(RF[f'rf_idx_L{l}'] == wi)[0][0])
    # pairs were built a-major, so reshaping gives [y1, y2]; .T puts y1 on x and y2 on y
    a = RF[f'rf_acts_L{l}'][:, col].reshape(len(grid), len(grid)).T
    v = np.abs(a).max()
    im = ax.imshow(a, origin='lower', extent=ext, aspect='auto', cmap='RdBu_r',
                   vmin=-v, vmax=v)
    ax.plot(ext[:2], ext[:2], '--', color='0.25', lw=1.0)          # y1 = y2
    ax.set_title(f'rank {r + 1}   L{l} n{wi}', fontsize=TICK_FS)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=9)
    if r % NCOL == 0:
        ax.set_ylabel('$y_2$', fontsize=LABEL_FS)
    if r // NCOL == nrow - 1:
        ax.set_xlabel('$y_1$', fontsize=LABEL_FS)
    ax.tick_params(labelsize=TICK_FS - 2)
fig.suptitle(f'neuron activation over the $(y_1, y_2)$ grid  (dashed: $y_1=y_2$)',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, f'{REGIME}_neuron_receptive_fields'); plt.show()

In [ ]:
SHOW_LAYERS = [14, 15]      # [14] for the L14 arrows alone
SHOW_WHICH  = ['gate', 'up']

LCOL = {14: COL['full'], 15: COL['das']}
WSTYLE = {'gate': dict(lw=2.4, ls='-'), 'up': dict(lw=1.6, ls=(0, (4, 2)))}

pq = TUNE['pq']
Rad = 0.92 * np.quantile(np.linalg.norm(pq, axis=1), 0.97)

fig, ax = plt.subplots(figsize=(8.4, 7.6))
ax.scatter(pq[:, 0], pq[:, 1], s=6, color='0.72', alpha=0.55, edgecolors='none',
           label='L14 cloud @ $n_2$', zorder=0)
t = np.linspace(-1, 1, 2) * Rad * 1.1
ax.plot(-TUNE['diff_axis'][1] * t, TUNE['diff_axis'][0] * t, '--', color='0.25', lw=1.2,
        zorder=1, label='$y_1=y_2$ boundary')

for which in SHOW_WHICH:
    for r in range(len(RF['rf_pooled'])):
        l = int(RF['rf_layer'][r])
        if l not in SHOW_LAYERS:
            continue
        d = RF[f'read_{which}'][r].astype(float)
        d = d / np.linalg.norm(d) * Rad          # direction only; magnitudes are in the table
        ax.annotate('', xy=d, xytext=(0, 0), zorder=3,
                    arrowprops=dict(arrowstyle='-|>', color=LCOL[l], alpha=0.9,
                                    shrinkA=0, shrinkB=0, **WSTYLE[which]))
        if which == SHOW_WHICH[0]:
            ax.text(*(d * 1.1), str(r + 1), color=LCOL[l], fontsize=TICK_FS,
                    ha='center', va='center', zorder=4,
                    bbox=dict(boxstyle='circle,pad=0.14', fc='white', ec='none', alpha=0.75))

for l in SHOW_LAYERS:
    ax.plot([], [], color=LCOL[l], lw=2.4, label=f'L{l} neuron read-in')
for which in SHOW_WHICH:
    ax.plot([], [], color='0.35', label=which, **WSTYLE[which])
ax.set_xlim(-Rad * 1.3, Rad * 1.3); ax.set_ylim(-Rad * 1.3, Rad * 1.3)
ax.set_aspect('equal')
ax.axhline(0, color='0.85', lw=0.8, zorder=0); ax.axvline(0, color='0.85', lw=0.8, zorder=0)
style_axes(ax, xlabel=r'$\mathbf{v}_1$ component  ($y_1$)',
           ylabel=r'$\mathbf{v}_2$ component  ($y_2$)', grid=False)
ax.legend(fontsize=LEGEND_FS - 2, frameon=False, loc='upper left')
ax.set_title(f'top-{RF_TOP} read-in directions in the plane (numbered by rank)',
             fontsize=TITLE_FS - 2)
plt.tight_layout(); savefig(fig, f'{REGIME}_neuron_read_directions'); plt.show()

## SVD of the L14 read-in weights inside the plane

The top-10's L14 neurons, each read-in vector $\gamma$-folded, unit-normalised and projected onto
span($\mathbf{v}_1,\mathbf{v}_2$), stacked into an $n\times2$ matrix and decomposed. Rows are
normalised by the **full** weight norm, so a row's length is the in-plane fraction and the SVD is
about how much of each neuron's reading lands in the plane, not about per-neuron weight scale.

Uncentred (the origin is meaningful — a zero row means the neuron does not read the plane at all),
so this is a second-moment decomposition, not a PCA. $\sigma_1/\sigma_2$ is the anisotropy: large
means the group reads one shared in-plane axis. Its null is a bootstrap over random L14 neurons
drawn from the alignment background, because even random vectors give $\sigma_1/\sigma_2>1$.

In [ ]:
SVD_LAYER  = 14
SVD_WHICH  = ['gate', 'up']
SVD_NBOOT  = 4000
ROW_NORM   = 'unit'     # 'unit' = rows carry the in-plane fraction; 'direction' = angles only


def inplane_unit(comp, frac):
    """Raw in-plane components -> rows of length `frac` (or unit length if ROW_NORM='direction')."""
    d = comp / np.linalg.norm(comp, axis=1, keepdims=True)
    return d if ROW_NORM == 'direction' else d * np.asarray(frac)[:, None]


def top_sv(Mx):
    """(sigma1, sigma2, first right singular vector oriented toward the difference axis)."""
    _u, s, vt = np.linalg.svd(Mx, full_matrices=False)
    v = vt[0]
    return s[0], s[1], (-v if v @ TUNE['diff_axis'] < 0 else v)


sel = RF['rf_layer'] == SVD_LAYER
rank_of = np.where(sel)[0] + 1
rng = np.random.default_rng(0)
print(f'L{SVD_LAYER} neurons in the top {len(RF["rf_pooled"])}: '
      + '  '.join(f'rank {r} n{i}' for r, i in zip(rank_of, RF['rf_within'][sel])))
print(f'difference axis {DIFF_ANG:+.1f} deg; rows normalised: {ROW_NORM}\n')

SVD = {}
for which in SVD_WHICH:
    Mx = inplane_unit(RF[f'read_{which}'][sel], RF[f'readfrac_{which}'][sel])
    s1, s2, v1 = top_sv(Mx)

    # Null: the same statistic on groups of the same size drawn from random L14 neurons, whose
    # in-plane fraction and angle the tuning cell already saved.
    bl = TUNE[f'align_lay_{ck(RF_CASE)}_bg_{which}'] == SVD_LAYER
    ba = np.radians(TUNE[f'align_ang_{ck(RF_CASE)}_bg_{which}'][bl])
    bf = TUNE[f'align_frac_{ck(RF_CASE)}_bg_{which}'][bl]
    bg = inplane_unit(np.c_[np.cos(ba), np.sin(ba)], bf)
    ratios, angs = np.empty(SVD_NBOOT), np.empty(SVD_NBOOT)
    for b in range(SVD_NBOOT):
        r1, r2, rv = top_sv(bg[rng.choice(len(bg), Mx.shape[0], replace=False)])
        ratios[b] = r1 / r2
        angs[b] = np.degrees(np.arctan2(rv[1], rv[0]))
    ratio, ang = s1 / s2, np.degrees(np.arctan2(v1[1], v1[0]))
    dev = abs((ang - DIFF_ANG + 90) % 180 - 90)
    nulldev = np.abs((angs - DIFF_ANG + 90) % 180 - 90)
    SVD[which] = dict(M=Mx, s1=s1, s2=s2, v1=v1, ratio=ratio, ang=ang, dev=dev,
                      pct=float((ratios < ratio).mean()), nulldev=float(np.median(nulldev)),
                      pctdev=float((nulldev < dev).mean()), ratios=ratios)

    print(f'--- {which}  ({Mx.shape[0]} x 2)')
    print(f'    singular values   s1 = {s1:.4f}   s2 = {s2:.4f}   s1/s2 = {ratio:.2f}'
          f'   s1^2 share = {s1**2 / (s1**2 + s2**2):.3f}')
    print(f'    right singular vectors   v1 = [{v1[0]:+.3f}, {v1[1]:+.3f}] at {ang:+.1f} deg'
          f'   v2 = [{-v1[1]:+.3f}, {v1[0]:+.3f}] at {ang + 90:+.1f} deg')
    print(f'    anisotropy vs null: {ratio:.2f} vs median {np.median(ratios):.2f} '
          f'(random L{SVD_LAYER} groups of {Mx.shape[0]}), percentile {SVD[which]["pct"]:.3f}')
    print(f'    v1 is {dev:.1f} deg off the difference axis; null median {SVD[which]["nulldev"]:.1f}'
          f' deg, percentile {SVD[which]["pctdev"]:.3f}')

In [ ]:
from matplotlib.patches import Ellipse

fig, axes = plt.subplots(1, len(SVD_WHICH), figsize=(6.6 * len(SVD_WHICH), 6.2), squeeze=False)
for ax, which in zip(axes[0], SVD_WHICH):
    B = SVD[which]
    Mx, v1, s1, s2 = B['M'], B['v1'], B['s1'], B['s2']
    lim = 1.35 * np.abs(Mx).max()

    ax.axhline(0, color='0.85', lw=0.8); ax.axvline(0, color='0.85', lw=0.8)
    t = np.linspace(-1, 1, 2) * lim
    ax.plot(TUNE['diff_axis'][0] * t, TUNE['diff_axis'][1] * t, '--', color='0.35', lw=1.2,
            label='difference axis')

    # The sigma-scaled ellipse: the shape of this group's reading of the plane.
    sc = lim / (1.25 * s1)
    ax.add_patch(Ellipse((0, 0), 2 * s1 * sc, 2 * s2 * sc,
                         angle=np.degrees(np.arctan2(v1[1], v1[0])),
                         fc=COL['full'], alpha=0.10, ec=COL['full'], lw=1.2, zorder=0))
    for k, (vec, sv, lab) in enumerate([(v1, s1, r'$\sigma_1$'),
                                        (np.array([-v1[1], v1[0]]), s2, r'$\sigma_2$')]):
        tip = vec * sv * sc
        ax.annotate('', xy=tip, xytext=(0, 0), zorder=4,
                    arrowprops=dict(arrowstyle='-|>', lw=3.0, color='#111111', shrinkA=0,
                                    shrinkB=0))
        ax.text(*(tip * 1.13), f'{lab} = {sv:.3f}', fontsize=TICK_FS, ha='center', va='center',
                zorder=5, bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none', alpha=0.8))

    for row, r in zip(Mx, rank_of):
        ax.annotate('', xy=row, xytext=(0, 0), zorder=3,
                    arrowprops=dict(arrowstyle='-|>', lw=1.8, color=COL['das'], alpha=0.85,
                                    shrinkA=0, shrinkB=0))
        ax.text(*(row * 1.12), str(r), color=COL['das'], fontsize=TICK_FS - 1, ha='center',
                va='center', zorder=4,
                bbox=dict(boxstyle='circle,pad=0.12', fc='white', ec='none', alpha=0.75))

    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    style_axes(ax, xlabel=r'$\mathbf{v}_1$ component', ylabel=r'$\mathbf{v}_2$ component',
               grid=False,
               title=f'{which}:  $\\sigma_1/\\sigma_2$ = {B["ratio"]:.2f} '
                     f'(null pct {B["pct"]:.2f}),  {B["dev"]:.0f}$\\degree$ off axis')
    ax.legend(fontsize=LEGEND_FS - 2, frameon=False, loc='upper left')
fig.suptitle(f'L{SVD_LAYER} read-in weights of the top-{len(RF["rf_pooled"])} neurons, '
             r'SVD inside span($\mathbf{v}_1,\mathbf{v}_2$)', fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_l{SVD_LAYER}_readin_svd'); plt.show()

## Pushing the plane through the neurons of interest

The same cloud, projected on the same signed span($\mathbf{v}_1,\mathbf{v}_2$) basis, read at each
stage:

**L14** — the plane is defined at the pre-MLP L14 residual, so MLP14's write is the direct image of
it: $m_{14}=\sum_{i\in\text{top L14}} a_i\,W^{\mathrm{down}}_{:,i}$.

**L15** — MLP15 reads the residual, which *already carries* the structure, and its write is added
on top. So the L15 row shows the pre-MLP L15 residual, MLP15's own write, and their sum.

Each panel is centred on its own mean (the mean in-plane write is printed instead), so the shape
and the $y_1\!-\!y_2$ separation are what is being compared, not the offset. All-neuron rows are a
control: they say whether any squashing is specific to the neurons of interest.

In [ ]:
PUSH_PATH = f'{RESULTS_DIR}/comparator_push_{REGIME}.npz'
PUSH_STAGES = ['pre14', 'm14_top', 'm14_all', 'post14_top',
               'pre15', 'm15_top', 'm15_all', 'post15_top']


def _push():
    i14 = torch.as_tensor(rf_within[rf_layer == 14], device=device, dtype=torch.long)
    i15 = torch.as_tensor(rf_within[rf_layer == 15], device=device, dtype=torch.long)
    sub = {14: i14, 15: i15}

    v1 = torch.tensor(h14_das_np.astype(np.float32), device=device)
    v2 = torch.tensor(l13_pc1b_np.astype(np.float32), device=device)
    e1 = v1 / v1.norm()
    e2 = v2 - (v2 @ e1) * e1
    e2 = e2 / e2.norm()
    sg = TUNE['sgn']
    E = torch.stack([e1 * float(sg[0]), e2 * float(sg[1])], dim=1)      # (d_model, 2)

    Wd = {l: mlp_of(l).down_proj.weight.detach().float()[:, sub[l]] for l in (14, 15)}
    pairs = np.array(S.sample_cloud_pairs(N_TUNE_CLOUD, SEED + 11, low, high), dtype=np.int64)

    store, acc = {}, {k: [] for k in PUSH_STAGES}

    def cap_resid(l):
        return D.model.model.layers[l].post_attention_layernorm.register_forward_pre_hook(
            lambda m, i, _l=l: store.__setitem__(f'x{_l}', i[0][:, POS_B, :].detach().float()))

    def cap_act(l):
        return mlp_of(l).down_proj.register_forward_pre_hook(
            lambda m, i, _l=l: store.__setitem__(f'a{_l}',
                                                 i[0][:, POS_B, sub[_l]].detach().float()))

    def cap_out(l):
        return mlp_of(l).register_forward_hook(
            lambda m, i, o, _l=l: store.__setitem__(f'o{_l}', o[:, POS_B, :].detach().float()))

    hs = [f(l) for l in (14, 15) for f in (cap_resid, cap_act, cap_out)]
    try:
        for i0 in range(0, len(pairs), EVAL_BS):
            ids = tokenizer([D.prompt_gen(list(p)) for p in pairs[i0:i0 + EVAL_BS]],
                            return_tensors='pt')['input_ids'].to(device)
            assert ids.shape[1] == T, 'cloud prompt length must match the regime template'
            with torch.no_grad():
                _ = D.model(ids)
            m14, m15 = store['a14'] @ Wd[14].T, store['a15'] @ Wd[15].T   # top-neuron writes only
            for k, v in (('pre14', store['x14']), ('m14_top', m14),
                         ('m14_all', store['o14']), ('post14_top', store['x14'] + m14),
                         ('pre15', store['x15']), ('m15_top', m15),
                         ('m15_all', store['o15']), ('post15_top', store['x15'] + m15)):
                acc[k].append((v @ E).cpu().numpy())
            store.clear()
    finally:
        for h in hs:
            h.remove()

    out = {'y1': pairs[:, 0].astype(np.float64), 'y2': pairs[:, 1].astype(np.float64)}
    for k in PUSH_STAGES:
        out[k] = np.concatenate(acc[k]).astype(np.float32)
    return out


PUSH = cached_npz(PUSH_PATH, _push)

ly1, ly2 = np.log(PUSH['y1']), np.log(PUSH['y2'])


def stage_stats(C):
    """Shape of a 2D stage cloud, and what a linear read of it recovers."""
    Cc = C - C.mean(0)
    s = np.linalg.svd(Cc, compute_uv=False)
    A = np.c_[Cc, np.ones(len(Cc))]

    def r2(y):
        res = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
        return 1 - res.var() / y.var()

    d = ly1 - ly2
    w = np.linalg.lstsq(A, d, rcond=None)[0][:2]
    proj = Cc @ (w / np.linalg.norm(w))
    pos, neg = proj[d > 0], proj[d < 0]
    auc = (pos[:, None] > neg[None, :]).mean()          # separability of y1 > y2 along that axis
    return s[0] / s[1], r2(ly1), r2(ly2), r2(d), max(auc, 1 - auc)


print(f'\n{"stage":>14}{"s1/s2":>8}{"R2 log y1":>11}{"R2 log y2":>11}{"R2 diff":>9}{"AUC y1>y2":>11}'
      f'{"mean write":>12}')
for k in PUSH_STAGES:
    a, r1, r2_, rd, auc = stage_stats(PUSH[k])
    mw = '' if k.startswith(('pre', 'post')) else f'{np.linalg.norm(PUSH[k].mean(0)):12.2f}'
    print(f'{k:>14}{a:8.2f}{r1:11.3f}{r2_:11.3f}{rd:9.3f}{auc:11.3f}{mw:>12}')

In [ ]:
PUSH_PANELS = [('pre14', 'pre-MLP L14 residual'),
               ('m14_top', f'MLP14 write, {int((rf_layer == 14).sum())} neurons'),
               ('post14_top', 'residual + MLP14 write'),
               ('pre15', 'pre-MLP L15 residual'),
               ('m15_top', f'MLP15 write, {int((rf_layer == 15).sum())} neurons'),
               ('post15_top', 'residual + MLP15 write')]

d = ly1 - ly2
vlim = np.abs(d).max()

# Every panel gets a square box and symmetric, equal x/y limits: the boxes are then identical in
# size and the shape of each cloud is still undistorted. (set_aspect('equal') instead lets each
# axes shrink to its own data range, which is what made the panels different sizes.) The units
# differ by panel -- a write is ~1, a residual ~20 -- so the tick labels carry the scale.
fig, axes = plt.subplots(2, 3, figsize=(14.5, 9.6), constrained_layout=True)
for ax, (key, lab) in zip(axes.ravel(), PUSH_PANELS):
    C = PUSH[key] - PUSH[key].mean(0)
    lim = 1.08 * np.abs(C).max()
    sc = ax.scatter(C[:, 0], C[:, 1], c=d, cmap='RdBu_r', vmin=-vlim, vmax=vlim, s=7,
                    alpha=0.85, edgecolors='none')
    # this stage's own y1 = y2 boundary: perpendicular to the in-plane gradient of log y1 - log y2
    w = np.linalg.lstsq(np.c_[C, np.ones(len(C))], d, rcond=None)[0][:2]
    w = w / np.linalg.norm(w)
    t = np.linspace(-1, 1, 2) * lim * 1.6
    ax.plot(-w[1] * t, w[0] * t, '--', color='0.25', lw=1.1)
    a, _, _, _, auc = stage_stats(PUSH[key])
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_box_aspect(1)
    ax.set_title(f'{lab}\n$\\sigma_1/\\sigma_2$={a:.1f}   AUC={auc:.2f}', fontsize=TICK_FS + 1)
    ax.tick_params(labelsize=TICK_FS - 3)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
axes[1, 1].set_xlabel(r'$\mathbf{v}_1$ component', fontsize=LABEL_FS)
for r in (0, 1):
    axes[r, 0].set_ylabel(r'$\mathbf{v}_2$ component', fontsize=LABEL_FS)
cb = fig.colorbar(sc, ax=axes, fraction=0.03, pad=0.015)
cb.set_label(r'$\log y_1 - \log y_2$', fontsize=LABEL_FS)
cb.ax.tick_params(labelsize=TICK_FS - 2)
fig.suptitle(r"the cloud in span($\mathbf{v}_1,\mathbf{v}_2$) pushed through the neurons of "
             r"interest   (panels centred; dashed = each stage's own $y_1=y_2$ boundary)",
             fontsize=TITLE_FS - 2)
savefig(fig, f'{REGIME}_plane_through_neurons'); plt.show()

## DAS inside the MLP's own neuron space

What is the MLP actually computing? Rank-$r$ DAS on the **activation vector of the neurons of
interest alone** — 6 dimensions at MLP14, 4 at MLP15 — rather than on the $d_{model}$ residual.
The interchange is
$a_S \leftarrow a_S + PP^{\!\top}(a_S^{\mathrm{clean}}-a_S)$ with $P$ a $k\times r$ orthonormal
basis, applied at the `down_proj` input at $n_2$.

Two things this buys over a residual-space DAS: the search space is 6- or 4-dimensional instead of
3584, and the learned direction is directly readable as **a weighting over named neurons**.
Full rank ($r=k$) is the ceiling — every one of those neurons set to its clean value.

In [ ]:
# MLPDAS_RANKS = [1, 2]
# MLPDAS_N, MLPDAS_STEPS, MLPDAS_LR, MLPDAS_BS = 64, 100, 0.05, 32
# MLPDAS_PATH = f'{RESULTS_DIR}/comparator_mlpdas_{REGIME}.npz'
# # 64 training triples, not the usual 128: the basis has only k*r free parameters here (at most 12),
# # so the counterfactual pairs are not the binding constraint and this halves the fit time.

# MLPDAS_SETS = {l: rf_within[rf_layer == l] for l in NEURON_LAYERS}


# def make_neuron_das_hook(idx, a_clean, P, pos):
#     """`down_proj` pre-hook: rank-r interchange inside the activations of `idx` at `pos`.

#     P is None -> full interchange of those neurons. Clones (unlike the freeze hook) because this
#     one runs under autograd, where an in-place write on a tensor the graph needs is invalid.
#     """
#     def _pre(mod, inp, _i=idx, _a=a_clean, _P=P, _p=pos):
#         x = inp[0].clone()
#         cur = x[:, _p, _i].float()
#         new = _a if _P is None else cur + ((_a - cur) @ _P) @ _P.T
#         x[:, _p, _i] = new.to(x.dtype)
#         return (x,) + inp[1:]
#     return _pre


# def train_neuron_das(layer, idx, a_clean, r, bat, seed,
#                      steps=MLPDAS_STEPS, lr=MLPDAS_LR, bs=MLPDAS_BS):
#     """Rank-r DAS inside the k-dimensional activation space of `idx` at layer `layer`."""
#     torch.manual_seed(seed)
#     das = D.DASSubspace(len(idx), r, device)
#     opt = torch.optim.Adam([das.raw], lr=lr)
#     bounds = [(i, min(i + bs, bat['n'])) for i in range(0, bat['n'], bs)]
#     curve = []
#     for _ in range(steps):
#         opt.zero_grad(set_to_none=True)
#         tot = 0.0
#         for lo, hi in bounds:
#             sub = _slice_batch(bat, lo, hi)
#             hnd = mlp_of(layer).down_proj.register_forward_pre_hook(
#                 make_neuron_das_hook(idx, a_clean[lo:hi], das.basis(), POS_B))
#             try:
#                 with torch.enable_grad():
#                     lg = D.model(sub['ids_r']).logits[:, -1, :].float()
#             finally:
#                 hnd.remove()
#             loss = D.cf_pair_loss(lg, sub) * (sub['n'] / bat['n'])
#             loss.backward()
#             tot += float(loss)
#             del lg
#         opt.step()
#         curve.append(tot)
#     torch.cuda.empty_cache()
#     with torch.no_grad():
#         return das.basis().detach(), np.array(curve, dtype=np.float32)


# def eval_neuron_das(layer, idx, a_clean, P, bat, bs=EVAL_BS):
#     recs, iias = [], []
#     for lo in range(0, bat['n'], bs):
#         hi = min(lo + bs, bat['n'])
#         sub = _slice_batch(bat, lo, hi)
#         hnd = mlp_of(layer).down_proj.register_forward_pre_hook(
#             make_neuron_das_hook(idx, a_clean[lo:hi], P, POS_B))
#         try:
#             with torch.no_grad():
#                 lg = D.model(sub['ids_r']).logits[:, -1, :].float()
#         finally:
#             hnd.remove()
#         r, i = posrec_and_iia(lg, sub)
#         recs.append(r); iias.append(i)
#         del lg
#     return np.concatenate(recs), np.concatenate(iias)


# def _mlpdas():
#     out, t0 = {}, time.time()
#     tri = pool[N_EVAL:][:MLPDAS_N]
#     bases = {}
#     for case in CASES:
#         tbat = make_batch(tri, CASE_SLOT[case], T)
#         for l, idx_np in MLPDAS_SETS.items():
#             idx = torch.as_tensor(idx_np, device=device, dtype=torch.long)
#             a_tr = capture_neurons(tbat['ids_c'], [l], POS_B, idx={l: idx})[l]
#             for r in MLPDAS_RANKS:
#                 P, curve = train_neuron_das(l, idx, a_tr, r, tbat, seed=SEED + r)
#                 bases[case, l, r] = P
#                 out[f'basis_{ck(case)}_L{l}_r{r}'] = P.cpu().numpy().astype(np.float32)
#                 out[f'curve_{ck(case)}_L{l}_r{r}'] = curve
#                 print(f'{case:>10} L{l} rank {r}: loss {curve[0]:.4f} -> {curve[-1]:.4f}'
#                       f'   ({time.time() - t0:.0f} s)', flush=True)
#             del a_tr
#             torch.cuda.empty_cache()

#     for ev in CASES:
#         for l, idx_np in MLPDAS_SETS.items():
#             idx = torch.as_tensor(idx_np, device=device, dtype=torch.long)
#             a_ev = capture_neurons(BAT[ev]['ids_c'], [l], POS_B, idx={l: idx})[l]
#             for tag, P in ([(f'r{r}', bases[ev, l, r]) for r in MLPDAS_RANKS] + [('full', None)]):
#                 rec, iia = eval_neuron_das(l, idx, a_ev, P, BAT[ev])
#                 out[f'posrec_{ck(ev)}_L{l}_{tag}'] = rec.astype(np.float32)
#                 out[f'iia_{ck(ev)}_L{l}_{tag}'] = iia.astype(np.float32)
#             del a_ev
#             torch.cuda.empty_cache()
#     out['n_used'] = np.array(len(tri))
#     print(f'total {time.time() - t0:.0f} s', flush=True)
#     return out


# MLPDAS = cached_npz(MLPDAS_PATH, _mlpdas)

# MLPDAS_TAGS = [f'r{r}' for r in MLPDAS_RANKS] + ['full']
# print(f'\n--- IIA, DAS inside the neuron space at n2   (n={N_EVAL})')
# print(f'{"case":>12}' + ''.join(f'{f"L{l} {t}":>10}' for l in MLPDAS_SETS for t in MLPDAS_TAGS))
# for case in CASES:
#     print(f'{case:>12}' + ''.join(f'{MLPDAS[f"iia_{ck(case)}_L{l}_{t}"].mean():10.3f}'
#                                   for l in MLPDAS_SETS for t in MLPDAS_TAGS))

In [ ]:
# MLPDAS_PLOT_CASE = CASES[0]

# # Cloud activations come from the tuning capture -- same cloud, so no extra forward pass.
# y1c = TUNE['y1']
# y2c = TUNE['y2']
# slot0 = y1c > y2c
# SLOT_COL = {0: COL['das'], 1: COL['full']}
# SLOT_LAB = {0: r'max in slot 1 ($y_1>y_2$)', 1: r'max in slot 2 ($y_2>y_1$)'}

# fig, axes = plt.subplots(len(MLPDAS_SETS), 3, figsize=(15.0, 4.7 * len(MLPDAS_SETS)),
#                          squeeze=False)
# for row, (l, idx_np) in enumerate(MLPDAS_SETS.items()):
#     cols = [int(np.where(TUNE[f'idx_L{l}'] == i)[0][0]) for i in idx_np]
#     A = TUNE[f'acts_L{l}'][:, cols]                       # (cloud, k) in the same neuron order
#     ranks = [int(np.where(RF['rf_within'] == i)[0][0]) + 1 for i in idx_np]

#     P1 = MLPDAS[f'basis_{ck(MLPDAS_PLOT_CASE)}_L{l}_r1']
#     c1 = (A - A.mean(0)) @ P1[:, 0]
#     ax = axes[row, 0]
#     bins = np.linspace(c1.min(), c1.max(), 55)
#     for s, m in ((0, slot0), (1, ~slot0)):
#         ax.hist(c1[m], bins=bins, color=SLOT_COL[s], alpha=0.7, label=SLOT_LAB[s])
#     style_axes(ax, xlabel='rank-1 DAS component', ylabel='cloud points',
#                title=f'MLP{l}, {len(idx_np)} neurons — rank 1')
#     if row == 0:
#         ax.legend(fontsize=LEGEND_FS - 3, frameon=False)

#     ax = axes[row, 1]
#     if 2 in MLPDAS_RANKS:
#         P2 = MLPDAS[f'basis_{ck(MLPDAS_PLOT_CASE)}_L{l}_r2']
#         C2 = (A - A.mean(0)) @ P2
#         for s, m in ((0, slot0), (1, ~slot0)):
#             ax.scatter(C2[m, 0], C2[m, 1], s=7, alpha=0.7, edgecolors='none', color=SLOT_COL[s])
#         ax.set_box_aspect(1)
#         style_axes(ax, xlabel='DAS 1', ylabel='DAS 2', title=f'MLP{l} — rank 2', grid=False)
#     else:
#         ax.axis('off')

#     ax = axes[row, 2]
#     ax.bar(np.arange(len(idx_np)), P1[:, 0], 0.68, color=COL['h14'], alpha=0.9,
#            edgecolor='white', linewidth=0)
#     ax.axhline(0, color='0.3', lw=0.8)
#     ax.set_xticks(np.arange(len(idx_np)))
#     ax.set_xticklabels([f'#{r}\nn{i}' for r, i in zip(ranks, idx_np)], fontsize=TICK_FS - 2)
#     style_axes(ax, xlabel='neuron (global rank, index)', ylabel='rank-1 DAS weight',
#                title=f'MLP{l} — what the direction uses')
# fig.suptitle(f'DAS inside the MLP neuron space (trained on "{MLPDAS_PLOT_CASE}"), '
#              'cloud coloured by which slot holds the max', fontsize=TITLE_FS - 2)
# plt.tight_layout(); savefig(fig, f'{REGIME}_mlp_neuron_das_space'); plt.show()

In [ ]:
# xg = np.arange(len(MLPDAS_TAGS))
# Wg = 0.8 / len(CASES)

# fig, axes = plt.subplots(1, len(MLPDAS_SETS), figsize=(7.2 * len(MLPDAS_SETS), 5.2), squeeze=False)
# for ax, (l, idx_np) in zip(axes[0], MLPDAS_SETS.items()):
#     for j, case in enumerate(CASES):
#         v = [MLPDAS[f'iia_{ck(case)}_L{l}_{t}'] for t in MLPDAS_TAGS]
#         m = np.array([x.mean() for x in v])
#         se = np.array([x.std() / np.sqrt(N_EVAL) for x in v])
#         b = ax.bar(xg + (j - (len(CASES) - 1) / 2) * Wg, m, Wg, yerr=se, capsize=3,
#                    color=[COL['das'], COL['full']][j], alpha=0.9, edgecolor='white',
#                    linewidth=0, label=case)
#         ax.bar_label(b, fmt='%.2f', fontsize=TICK_FS - 3, padding=2)
#     ax.set_ylim(0, 1.05)
#     ax.set_xticks(xg)
#     ax.set_xticklabels([t.replace('r', 'rank ') if t != 'full' else f'full ({len(idx_np)}d)'
#                         for t in MLPDAS_TAGS], fontsize=TICK_FS)
#     style_axes(ax, ylabel='IIA', title=f'MLP{l}, {len(idx_np)} neurons')
# axes[0, 0].legend(fontsize=LEGEND_FS, frameon=False, loc='upper left')
# fig.suptitle('DAS inside the MLP neuron space, trained and evaluated within each case',
#              fontsize=TITLE_FS - 1)
# plt.tight_layout(); savefig(fig, f'{REGIME}_mlp_neuron_das_iia'); plt.show()

## DAS at the L15 residual, rank 1 / 2 / 3

Rank-$k$ DAS trained on the residual stream leaving layer 15 at $n_2$ — the site whose write §19a
showed to be a two-cluster, rank-1 code. Same objective and optimiser as the rest of the project
(`cf_pair_loss`, QR retraction, 128 held-out triples, 100 steps, lr 0.05), trained **separately per
case** so each bar is its own experiment, then cross-evaluated.

Cloud points are coloured by **which slot holds the maximum**, the variable DAS is being asked to
control.

In [ ]:
DAS15_LAYER = 15
DAS15_RANKS = [1, 2, 3]
DAS15_STEPS, DAS15_LR, DAS15_BS = 100, 0.05, 32
DAS15_PATH = f'{RESULTS_DIR}/comparator_das15_{REGIME}.npz'


def train_das_at(layer, pos, k, bat, ch, seed, steps=DAS15_STEPS, lr=DAS15_LR, bs=DAS15_BS):
    """Rank-k DAS on the residual leaving `layer` at `pos`. Mirrors `data_numberreps.train_das_1d`
    (which is pinned to L13) and `data_sharedrep.train_h14_das`: QR-retraction basis, counterfactual
    pair cross-entropy, gradients accumulated over minibatches so the training-set size is
    decoupled from the backward pass's activation memory."""
    torch.manual_seed(seed)
    das = D.DASSubspace(d_model, k, device)
    opt = torch.optim.Adam([das.raw], lr=lr)
    bounds = [(i, min(i + bs, bat['n'])) for i in range(0, bat['n'], bs)]
    curve = []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        tot = 0.0
        for lo, hi in bounds:
            sub = _slice_batch(bat, lo, hi)
            hnd = D.model.model.layers[layer].register_forward_hook(
                D.make_subspace_patch_hook(das.basis(), ch[lo:hi], pos))
            try:
                with torch.enable_grad():
                    lg = D.model(sub['ids_r']).logits[:, -1, :].float()
            finally:
                hnd.remove()
            loss = D.cf_pair_loss(lg, sub) * (sub['n'] / bat['n'])
            loss.backward()
            tot += float(loss)
            del lg
        opt.step()
        curve.append(tot)
    torch.cuda.empty_cache()
    with torch.no_grad():
        return das.basis().detach(), np.array(curve, dtype=np.float32)


def _das15():
    out, t0 = {}, time.time()
    bases = {}
    for case in CASES:
        tbat = make_batch(pool[N_EVAL:], CASE_SLOT[case], T)
        ch = S.clean_hidden_at(tbat['ids_c'], DAS15_LAYER)
        for k in DAS15_RANKS:
            Q, curve = train_das_at(DAS15_LAYER, POS_B, k, tbat, ch, seed=SEED + k)
            bases[case, k] = Q
            out[f'basis_{ck(case)}_k{k}'] = Q.cpu().numpy().astype(np.float32)
            out[f'curve_{ck(case)}_k{k}'] = curve
            print(f'{case:>10} rank {k}: loss {curve[0]:.4f} -> {curve[-1]:.4f}'
                  f'   ({time.time() - t0:.0f} s elapsed)', flush=True)
        del ch
        torch.cuda.empty_cache()

    # IIA / posrec of every basis in every case, plus the full-rank interchange at the same site.
    for ev in CASES:
        src = {DAS15_LAYER: S.clean_hidden_at(BAT[ev]['ids_c'], DAS15_LAYER)}
        for tr in CASES:
            for k in DAS15_RANKS:
                spec = dict(resid=[(DAS15_LAYER, 'sub', bases[tr, k])])
                rec, iia = eval_spec(spec, BAT[ev], src, POS_B)
                out[f'posrec_{ck(ev)}_from_{ck(tr)}_k{k}'] = rec.astype(np.float32)
                out[f'iia_{ck(ev)}_from_{ck(tr)}_k{k}'] = iia.astype(np.float32)
        rec, iia = eval_spec(dict(resid=[(DAS15_LAYER, 'full', None)]), BAT[ev], src, POS_B)
        out[f'posrec_{ck(ev)}_full'] = rec.astype(np.float32)
        out[f'iia_{ck(ev)}_full'] = iia.astype(np.float32)
        del src
        torch.cuda.empty_cache()

    # The cloud in each subspace: residual leaving L15 at n2, projected onto every basis.
    pairs = np.array(S.sample_cloud_pairs(N_TUNE_CLOUD, SEED + 11, low, high), dtype=np.int64)
    store, chunks = {}, []
    hnd = D.model.model.layers[DAS15_LAYER].register_forward_hook(
        lambda m, i, o: store.__setitem__('h', (o[0] if isinstance(o, tuple) else o)
                                          [:, POS_B, :].detach().float()))
    try:
        for i0 in range(0, len(pairs), EVAL_BS):
            ids = tokenizer([D.prompt_gen(list(p)) for p in pairs[i0:i0 + EVAL_BS]],
                            return_tensors='pt')['input_ids'].to(device)
            with torch.no_grad():
                _ = D.model(ids)
            chunks.append(store.pop('h'))
    finally:
        hnd.remove()
    H = torch.cat(chunks, dim=0)
    for case in CASES:
        for k in DAS15_RANKS:
            out[f'proj_{ck(case)}_k{k}'] = (H @ bases[case, k]).cpu().numpy().astype(np.float32)
    out['cloud_y1'], out['cloud_y2'] = pairs[:, 0].astype(np.int64), pairs[:, 1].astype(np.int64)
    del H
    torch.cuda.empty_cache()
    print(f'total {time.time() - t0:.0f} s', flush=True)
    return out


DAS15 = cached_npz(DAS15_PATH, _das15)

print(f'\n--- IIA at the L{DAS15_LAYER} residual @ n2   (n={N_EVAL})')
print(f'{"evaluated in":>14}' + ''.join(f'{f"r{k} own":>10}' for k in DAS15_RANKS)
      + ''.join(f'{f"r{k} xfer":>10}' for k in DAS15_RANKS) + f'{"full":>10}')
for ev in CASES:
    tr_other = CASES[1 - CASES.index(ev)]
    own = ''.join(f'{DAS15[f"iia_{ck(ev)}_from_{ck(ev)}_k{k}"].mean():10.3f}' for k in DAS15_RANKS)
    xf = ''.join(f'{DAS15[f"iia_{ck(ev)}_from_{ck(tr_other)}_k{k}"].mean():10.3f}'
                 for k in DAS15_RANKS)
    print(f'{ev:>14}{own}{xf}{DAS15[f"iia_{ck(ev)}_full"].mean():10.3f}')

In [ ]:
DAS15_PLOT_CASE = CASES[0]        # whose basis the cloud is shown in

y1c, y2c = DAS15['cloud_y1'].astype(float), DAS15['cloud_y2'].astype(float)
slot0 = y1c > y2c                 # True: the maximum sits in the first slot
SLOT_COL = {0: COL['das'], 1: COL['full']}
SLOT_LAB = {0: r'max in slot 1 ($y_1>y_2$)', 1: r'max in slot 2 ($y_2>y_1$)'}

fig = plt.figure(figsize=(16.5, 5.2))
for j, k in enumerate(DAS15_RANKS):
    P = DAS15[f'proj_{ck(DAS15_PLOT_CASE)}_k{k}']
    P = P - P.mean(0)
    if k == 1:
        ax = fig.add_subplot(1, 3, 1)
        bins = np.linspace(P.min(), P.max(), 60)
        for s, m in ((0, slot0), (1, ~slot0)):
            ax.hist(P[m, 0], bins=bins, color=SLOT_COL[s], alpha=0.7, label=SLOT_LAB[s])
        style_axes(ax, xlabel='DAS component', ylabel='cloud points')
        ax.legend(fontsize=LEGEND_FS - 2, frameon=False)
    elif k == 2:
        ax = fig.add_subplot(1, 3, 2)
        for s, m in ((0, slot0), (1, ~slot0)):
            ax.scatter(P[m, 0], P[m, 1], s=7, alpha=0.7, edgecolors='none', color=SLOT_COL[s])
        style_axes(ax, xlabel='DAS 1', ylabel='DAS 2', grid=False)
        ax.set_box_aspect(1)
    else:
        ax = fig.add_subplot(1, 3, 3, projection='3d')
        for s, m in ((0, slot0), (1, ~slot0)):
            ax.scatter(P[m, 0], P[m, 1], P[m, 2], s=5, alpha=0.6, edgecolors='none',
                       color=SLOT_COL[s])
        ax.set_xlabel('DAS 1', fontsize=TICK_FS); ax.set_ylabel('DAS 2', fontsize=TICK_FS)
        ax.set_zlabel('DAS 3', fontsize=TICK_FS)
        ax.tick_params(labelsize=TICK_FS - 3)
        ax.view_init(elev=22, azim=-58)
    ax.set_title(f'rank {k}', fontsize=TITLE_FS)
fig.suptitle(f'L{DAS15_LAYER} residual @ $n_2$ in the DAS subspace '
             f'(trained on "{DAS15_PLOT_CASE}"), coloured by which slot holds the max',
             fontsize=TITLE_FS - 2)
plt.tight_layout(); savefig(fig, f'{REGIME}_l{DAS15_LAYER}_das_space'); plt.show()

In [ ]:
DAS15_SHOW_POSREC = True

groups = [f'rank {k}' for k in DAS15_RANKS] + ['full rank']
xg = np.arange(len(groups))
Wg = 0.8 / len(CASES)
mets = [('iia', 'IIA', (0, 1.05))] + ([('posrec', 'position recovery', None)]
                                      if DAS15_SHOW_POSREC else [])

fig, axes = plt.subplots(1, len(mets), figsize=(7.6 * len(mets), 5.4), squeeze=False)
for ax, (metric, ylab, ylim) in zip(axes[0], mets):
    for j, case in enumerate(CASES):
        v = [DAS15[f'{metric}_{ck(case)}_from_{ck(case)}_k{k}'] for k in DAS15_RANKS]
        v.append(DAS15[f'{metric}_{ck(case)}_full'])
        m = np.array([x.mean() for x in v])
        se = np.array([x.std() / np.sqrt(N_EVAL) for x in v])
        b = ax.bar(xg + (j - (len(CASES) - 1) / 2) * Wg, m, Wg, yerr=se, capsize=3,
                   color=[COL['das'], COL['full']][j], alpha=0.9, edgecolor='white',
                   linewidth=0, label=case)
        ax.bar_label(b, fmt='%.2f', fontsize=TICK_FS - 3, padding=2)
    if ylim: ax.set_ylim(*ylim)
    ax.axhline(0, color='0.3', lw=0.8)
    ax.set_xticks(xg); ax.set_xticklabels(groups, fontsize=TICK_FS)
    style_axes(ax, ylabel=ylab)
axes[0, 0].legend(fontsize=LEGEND_FS, frameon=False, loc='upper left')
fig.suptitle(f'DAS at the L{DAS15_LAYER} residual @ $n_2$, trained and evaluated within each case',
             fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_l{DAS15_LAYER}_das_iia'); plt.show()

## Which MLP15 neurons write along the L15 DAS direction, and what feeds them

Two questions, both about the neurons of interest rather than the residual as a whole.

**Alignment.** MLP15 neuron $j$ writes into the residual along $W^{\mathrm{down}}_{:,j}$, and the
L15 DAS basis $Q$ is a subspace of that same residual, so the natural statistic is the fraction of
the write that lands inside it, $\lVert Q^{\!\top}w_j\rVert/\lVert w_j\rVert$ (for rank 1 this is
$|\cos|$). Significance comes from the empirical null over all 18944 MLP15 neurons, not from a
$\chi^2$. A second, functional column weights that by how much the neuron actually varies.

**Connectivity.** L14 neuron $i$ writes $W^{\mathrm{down}}_{:,i}$; L15 neuron $j$ reads the residual
through its $\gamma$-folded `gate` and `up` rows. Their inner product is the direct weight path
(the "virtual weight" of Elhage et al. 2021). That is geometry only, so it is paired with a causal
edge: patch L14 neuron $i$ to its clean value and measure how far each L15 neuron moves, in units
of its own spread.

In [ ]:
CONNECT_PATH = f'{RESULTS_DIR}/comparator_connect_{REGIME}.npz'
I14 = rf_within[rf_layer == 14]
I15 = rf_within[rf_layer == 15]
RANK14 = [int(np.where(RF['rf_within'] == i)[0][0]) + 1 for i in I14]
RANK15 = [int(np.where(RF['rf_within'] == i)[0][0]) + 1 for i in I15]


def _connect():
    out = {}
    t14 = torch.as_tensor(I14, device=device, dtype=torch.long)
    t15 = torch.as_tensor(I15, device=device, dtype=torch.long)
    Wd14 = mlp_of(14).down_proj.weight.detach().float()            # (d_model, d_ff), col = write
    Wd15 = mlp_of(15).down_proj.weight.detach().float()
    g15 = D.model.model.layers[15].post_attention_layernorm.weight.detach().float()
    reads = {'gate': mlp_of(15).gate_proj.weight.detach().float() * g15,     # (d_ff, d_model)
             'up': mlp_of(15).up_proj.weight.detach().float() * g15}

    # -- Q1: how much of each MLP15 neuron's write lies in the L15 DAS subspace ----------------
    # Computed for every neuron, so the four of interest can be placed in their own null.
    nrm15 = Wd15.norm(dim=0)
    for case in CASES:
        for k in DAS15_RANKS:
            Q = torch.tensor(DAS15[f'basis_{ck(case)}_k{k}'], device=device)
            out[f'dasfrac_{ck(case)}_k{k}'] = ((Q.T @ Wd15).norm(dim=0) / nrm15).cpu().numpy()

    # -- Q2a: direct weight path, L14 write -> L15 read ----------------------------------------
    A = Wd14[:, t14]                                               # (d_model, 6)
    An = A / A.norm(dim=0, keepdim=True)
    for which, R in reads.items():
        Rn = R / R.norm(dim=1, keepdim=True)                       # (d_ff, d_model)
        allc = (Rn @ An).cpu().numpy()                             # (d_ff, 6) -- the null
        out[f'path_{which}'] = allc[I15].T.astype(np.float32)      # (6, 4) pairs of interest
        out[f'path_{which}_null'] = allc.astype(np.float32)

    # -- Q2b: causal edge. Patch one L14 neuron, read the L15 neurons --------------------------
    sd15 = TUNE[f'acts_L15'][:, [int(np.where(TUNE['idx_L15'] == i)[0][0]) for i in I15]].std(0)
    out['sd15'] = sd15.astype(np.float32)
    for case in CASES:
        bat = BAT[case]
        a_cl = capture_neurons(bat['ids_c'], [14], POS_B, idx={14: t14})[14]
        base = capture_neurons(bat['ids_r'], [15], POS_B, idx={15: t15})[15].cpu().numpy()
        rows = []
        for c, _ in enumerate(I14) :
            one = t14[c:c + 1]
            got = capture_neurons(
                bat['ids_r'], [15], POS_B, idx={15: t15},
                patch=lambda lo, hi, _o=one, _c=c: [mlp_of(14).down_proj.register_forward_pre_hook(
                    make_neuron_freeze_hook(_o, a_cl[lo:hi, _c:_c + 1], POS_B))])[15].cpu().numpy()
            rows.append((got - base).mean(0) / sd15)
        got = capture_neurons(
            bat['ids_r'], [15], POS_B, idx={15: t15},
            patch=lambda lo, hi: [mlp_of(14).down_proj.register_forward_pre_hook(
                make_neuron_freeze_hook(t14, a_cl[lo:hi], POS_B))])[15].cpu().numpy()
        out[f'edge_{ck(case)}'] = np.array(rows, dtype=np.float32)          # (6, 4), in sd units
        out[f'edge_all_{ck(case)}'] = ((got - base).mean(0) / sd15).astype(np.float32)
        del a_cl
        torch.cuda.empty_cache()
    return out


CONNECT = cached_npz(CONNECT_PATH, _connect)

print(f'\n--- Q1: MLP15 write inside the L{DAS15_LAYER} DAS subspace  (null = all {D_FF or 18944} '
      f'MLP15 neurons)')
for case in CASES:
    print(f'  {case}')
    print(f'{"neuron":>10}{"rank":>6}' + ''.join(f'{f"r{k} frac":>10}{f"pct":>7}'
                                                 for k in DAS15_RANKS) + f'{"sd*|cos|":>10}')
    for c, (nrn, rk) in enumerate(zip(I15, RANK15)):
        cells = ''
        for k in DAS15_RANKS:
            f = CONNECT[f'dasfrac_{ck(case)}_k{k}']
            cells += f'{f[nrn]:10.4f}{(f < f[nrn]).mean():7.3f}'
        f1 = CONNECT[f'dasfrac_{ck(case)}_k1'][nrn]
        print(f'{"n" + str(nrn):>10}{rk:>6}{cells}{f1 * CONNECT["sd15"][c]:10.3f}')

print(f'\n--- Q2a: direct weight path, |cos| of L14 write with L15 read '
      f'(null percentile in brackets)')
for which in ('gate', 'up'):
    M, N = CONNECT[f'path_{which}'], CONNECT[f'path_{which}_null']
    print(f'  {which}:  rows = L14 neurons, cols = L15 neurons')
    print(f'{"":>12}' + ''.join(f'{f"#{r} n{n}":>16}' for r, n in zip(RANK15, I15)))
    for a, (na, ra) in enumerate(zip(I14, RANK14)):
        row = ''.join(f'{M[a, b]:+.3f} [{(np.abs(N[:, a]) < abs(M[a, b])).mean():.2f}]'.rjust(16)
                      for b in range(len(I15)))
        print(f'{f"#{ra} n{na}":>12}{row}')

print(f'\n--- Q2b: causal edge, patch one L14 neuron -> shift of each L15 neuron (in its own SD)')
for case in CASES:
    E = CONNECT[f'edge_{ck(case)}']
    print(f'  {case}')
    print(f'{"":>12}' + ''.join(f'{f"#{r} n{n}":>11}' for r, n in zip(RANK15, I15)))
    for a, (na, ra) in enumerate(zip(I14, RANK14)):
        print(f'{f"#{ra} n{na}":>12}' + ''.join(f'{E[a, b]:+11.2f}' for b in range(len(I15))))
    print(f'{"all 6":>12}' + ''.join(f'{v:+11.2f}' for v in CONNECT[f'edge_all_{ck(case)}']))

In [ ]:
fig, axes = plt.subplots(1, len(CASES), figsize=(7.2 * len(CASES), 5.0), squeeze=False)
for ax, case in zip(axes[0], CASES):
    f = CONNECT[f'dasfrac_{ck(case)}_k1']
    q95, q99 = np.quantile(f, [0.95, 0.99])
    ax.axhspan(0, q95, color='0.88', zorder=0, label='null, below 95th pct')
    ax.axhline(q99, ls=':', color='0.45', lw=1.1, label='null 99th pct')
    b = ax.bar(np.arange(len(I15)), [f[n] for n in I15], 0.6, color=COL['h14'], alpha=0.9,
               edgecolor='white', linewidth=0)
    ax.bar_label(b, labels=[f'pct {(f < f[n]).mean():.3f}' for n in I15], fontsize=TICK_FS - 2,
                 padding=3)
    ax.set_xticks(np.arange(len(I15)))
    ax.set_xticklabels([f'#{r}\nn{n}' for r, n in zip(RANK15, I15)], fontsize=TICK_FS)
    style_axes(ax, ylabel=r'$\|Q^\top w_j\| / \|w_j\|$  (rank-1 DAS)',
               xlabel='MLP15 neuron (global rank, index)', title=case)
axes[0, 0].legend(fontsize=LEGEND_FS - 2, frameon=False, loc='upper left')
fig.suptitle(f'how much of each MLP15 neuron\'s write lies along the L{DAS15_LAYER} DAS direction',
             fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_l15_neuron_das_alignment'); plt.show()

In [ ]:
xl = [f'#{r}\nn{n}' for r, n in zip(RANK15, I15)]
yl = [f'#{r} n{n}' for r, n in zip(RANK14, I14)]
MATS = ([(f'path_{w}', f'weight path, {w}', 'PuOr_r', None) for w in ('gate', 'up')]
        + [(f'edge_{ck(c)}', f'causal edge, {c}', 'RdBu_r', None) for c in CASES])

fig, axes = plt.subplots(2, 2, figsize=(12.4, 10.4))
for ax, (key, lab, cmap, _) in zip(axes.ravel(), MATS):
    M = CONNECT[key]
    v = np.abs(M).max()
    im = ax.imshow(M, cmap=cmap, vmin=-v, vmax=v, aspect='auto')
    for a in range(M.shape[0]):
        for b in range(M.shape[1]):
            ax.text(b, a, f'{M[a, b]:+.2f}', ha='center', va='center', fontsize=TICK_FS - 1,
                    color='white' if abs(M[a, b]) > 0.62 * v else 'black')
    ax.set_xticks(range(len(xl))); ax.set_xticklabels(xl, fontsize=TICK_FS - 1)
    ax.set_yticks(range(len(yl))); ax.set_yticklabels(yl, fontsize=TICK_FS - 1)
    ax.set_title(lab, fontsize=TITLE_FS - 2)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=9)
fig.supxlabel('MLP15 neuron', fontsize=LABEL_FS)
fig.supylabel('MLP14 neuron', fontsize=LABEL_FS)
fig.suptitle('L14 -> L15 neuron connectivity: direct weight path (top) and causal edge in SD '
             'units (bottom)', fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_l14_l15_connectivity'); plt.show()

## Each L14 neuron's switching boundary in the $\mathbf{v}_1,\mathbf{v}_2$ plane

One panel per L14 neuron of interest: the cloud coloured by that neuron's activation, its
$\gamma$-folded `gate` read direction as an arrow, and **two** versions of where it switches on.

A SwiGLU neuron is off where its gate preactivation $z=w_{\mathrm{eff}}\!\cdot\!x$ is negative, so
the boundary is a hyperplane through the origin in $d_{model}$ dimensions. (RMSNorm rescales $z$ by
a positive per-token scalar, which cannot move the boundary.) A hyperplane does not *project* to a
line — it projects onto the whole plane — so two different well-defined objects are drawn:

- **slice** (solid): the hyperplane's trace at the cloud's mean out-of-plane state,
  $ap+bq+c_0=0$ with $(a,b)$ the in-plane gradient. This is **exactly orthogonal to the arrow**,
  and is the boundary a point would see if only its in-plane coordinates changed.
- **fit** (dash-dot): the least-squares fit of $z$ on $(p,q)$ over the cloud. Its normal is a total
  regression coefficient, not a partial derivative, so it absorbs out-of-plane variation that is
  correlated with $p$ and $q$ — and is therefore *not* orthogonal to the arrow in general.

The gap between the two lines is the effect of that correlated out-of-plane structure. Each panel
reports both lines' sign accuracy against the true switch, plus the fit's $R^2$.

In [ ]:
import torch.nn.functional as Fn

BOUND_PATH = f'{RESULTS_DIR}/comparator_bound_{REGIME}.npz'
BOUND_LAYER = 14


def _bound():
    idx_np = rf_within[rf_layer == BOUND_LAYER]
    idx = torch.as_tensor(idx_np, device=device, dtype=torch.long)
    pairs = np.array(S.sample_cloud_pairs(N_TUNE_CLOUD, SEED + 11, low, high), dtype=np.int64)

    store, gs, us, xs = {}, [], [], []
    hs = [mlp_of(BOUND_LAYER).gate_proj.register_forward_hook(
              lambda m, i, o: store.__setitem__('g', o[:, POS_B, idx].detach().float())),
          mlp_of(BOUND_LAYER).up_proj.register_forward_hook(
              lambda m, i, o: store.__setitem__('u', o[:, POS_B, idx].detach().float())),
          D.model.model.layers[BOUND_LAYER].post_attention_layernorm.register_forward_pre_hook(
              lambda m, i: store.__setitem__('x', i[0][:, POS_B, :].detach().float()))]
    try:
        for i0 in range(0, len(pairs), EVAL_BS):
            ids = tokenizer([D.prompt_gen(list(p)) for p in pairs[i0:i0 + EVAL_BS]],
                            return_tensors='pt')['input_ids'].to(device)
            with torch.no_grad():
                _ = D.model(ids)
            gs.append(store.pop('g')); us.append(store.pop('u')); xs.append(store.pop('x'))
    finally:
        for h in hs:
            h.remove()
    G = torch.cat(gs).cpu().numpy()                       # gate preactivation, (cloud, 6)
    U = torch.cat(us).cpu().numpy()
    A = (Fn.silu(torch.tensor(G)) * torch.tensor(U)).numpy()
    mean_x = torch.cat(xs).mean(0)                        # cloud mean pre-MLP residual

    # -- the hyperplane's trace at the mean out-of-plane state -------------------------------
    # z = w.x is exactly (in-plane gradient).(p, q) + w.(out-of-plane part); fixing the latter at
    # its cloud mean leaves a line whose normal IS the in-plane gradient, i.e. the drawn arrow.
    # RMSNorm only scales z by a positive number, so it cannot move this boundary.
    v1 = torch.tensor(h14_das_np.astype(np.float32), device=device)
    v2 = torch.tensor(l13_pc1b_np.astype(np.float32), device=device)
    e1 = v1 / v1.norm()
    e2 = v2 - (v2 @ e1) * e1
    e2 = e2 / e2.norm()
    sg = TUNE['sgn']
    E = torch.stack([e1 * float(sg[0]), e2 * float(sg[1])], dim=1)
    g15 = D.model.model.layers[BOUND_LAYER].post_attention_layernorm.weight.detach().float()
    W = mlp_of(BOUND_LAYER).gate_proj.weight.detach()[idx].float() * g15      # (6, d_model)
    grad = (W @ E).cpu().numpy()                                             # (6, 2)
    c0 = (W @ mean_x).cpu().numpy()                                          # (6,)

    pq = TUNE['pq'].astype(np.float64)
    pred_slice = pq @ grad.T + c0
    acc_slice = ((pred_slice > 0) == (G > 0)).mean(0)

    # -- the best planar predictor of z (absorbs correlated out-of-plane variation) -----------
    X = np.c_[pq, np.ones(len(pq))]
    coef = np.linalg.lstsq(X, G.astype(np.float64), rcond=None)[0]           # (3, 6)
    pred = X @ coef
    r2 = 1 - ((G - pred) ** 2).sum(0) / ((G - G.mean(0)) ** 2).sum(0)
    acc = ((pred > 0) == (G > 0)).mean(0)
    return {'gate_pre': G.astype(np.float32), 'up_pre': U.astype(np.float32),
            'act': A.astype(np.float32), 'coef': coef.astype(np.float32),
            'grad': grad.astype(np.float32), 'c0': c0.astype(np.float32),
            'r2': r2.astype(np.float32), 'acc': acc.astype(np.float32),
            'acc_slice': acc_slice.astype(np.float32), 'idx': idx_np.astype(np.int64),
            'on_frac': (G > 0).mean(0).astype(np.float32)}


BOUND = cached_npz(BOUND_PATH, _bound)
if 'grad' not in BOUND:              # cache written before the slice boundary was added
    os.remove(BOUND_PATH)
    print('stale cache (no slice boundary) removed -- recomputing')
    BOUND = cached_npz(BOUND_PATH, _bound)

# The slice gradient must reproduce the arrow drawn in the earlier figures.
_chk = np.abs(BOUND['grad'] - RF['read_gate'][rf_layer == BOUND_LAYER]).max()
print(f'slice gradient vs the read_gate arrow: max abs difference {_chk:.2e}')

# "Always on" is a perfect classifier for a neuron that never switches, so every accuracy is
# quoted against the majority baseline max(on, 1-on). An accuracy at the baseline means the
# boundary does not usefully cut this cloud, whatever angle it was drawn at.
BOUND_BASE = np.maximum(BOUND['on_frac'], 1 - BOUND['on_frac'])

print(f'\n--- L{BOUND_LAYER} neurons: where the gate switches, in span(v1, v2)')
print(f'{"rank":>6}{"neuron":>9}{"gate on":>9}{"baseline":>10}{"slice acc":>11}{"fit acc":>9}'
      f'{"fit R2":>8}{"slice angle":>13}{"fit angle":>11}{"slice-fit":>11}')
for c, n in enumerate(BOUND['idx']):
    rk = int(np.where(RF['rf_within'] == n)[0][0]) + 1
    sa, sb = BOUND['grad'][c]
    fa, fb = BOUND['coef'][0, c], BOUND['coef'][1, c]
    ang = lambda a, b: np.degrees(np.arctan2(a, -b))          # the line, normal to (a, b)
    gap = abs((ang(sa, sb) - ang(fa, fb) + 90) % 180 - 90)
    print(f'{rk:>6}{"n" + str(n):>9}{BOUND["on_frac"][c]:9.3f}{BOUND_BASE[c]:10.3f}'
          f'{BOUND["acc_slice"][c]:11.3f}{BOUND["acc"][c]:9.3f}{BOUND["r2"][c]:8.3f}'
          f'{ang(sa, sb):>12.1f}d{ang(fa, fb):>10.1f}d{gap:>10.1f}d')
print('\nan accuracy at the baseline = the boundary does not cut this cloud usefully;\n'
      'a low fit R2 = the plane barely predicts z, so the fitted line\'s angle is not meaningful.')

In [ ]:
BOUND_NCOL = 3
BOUND_ARROW = 'gate'        # read direction drawn as the arrow

pq = TUNE['pq']
n_show = len(BOUND['idx'])
nrow = int(np.ceil(n_show / BOUND_NCOL))
lim = 1.12 * np.quantile(np.linalg.norm(pq, axis=1), 0.995)
read = RF[f'read_{BOUND_ARROW}'][rf_layer == BOUND_LAYER]


def line_pts(a, b, c, L):
    """The line a*x + b*y + c = 0, clipped to a box of half-width L."""
    if abs(b) > abs(a):
        x = np.array([-L, L]); return x, -(a * x + c) / b
    y = np.array([-L, L]); return -(b * y + c) / a, y


fig, axes = plt.subplots(nrow, BOUND_NCOL, figsize=(5.0 * BOUND_NCOL, 4.9 * nrow), squeeze=False)
for c in range(nrow * BOUND_NCOL):
    ax = axes[c // BOUND_NCOL, c % BOUND_NCOL]
    if c >= n_show:
        ax.axis('off')
        continue
    n = int(BOUND['idx'][c])
    rk = int(np.where(RF['rf_within'] == n)[0][0]) + 1
    a_act = BOUND['act'][:, c]
    v = np.abs(a_act).max()
    sc = ax.scatter(pq[:, 0], pq[:, 1], c=a_act, cmap='RdBu_r', vmin=-v, vmax=v, s=6,
                    alpha=0.85, edgecolors='none', zorder=1)

    t = np.linspace(-1, 1, 2) * lim * 1.6
    ax.plot(-TUNE['diff_axis'][1] * t, TUNE['diff_axis'][0] * t, '--', color='0.45', lw=1.1,
            zorder=2, label='$y_1=y_2$')
    ax.plot(*line_pts(*BOUND['grad'][c].astype(float), float(BOUND['c0'][c]), lim), '-',
            color='#111111', lw=2.0, zorder=3, label='gate $z=0$, mean slice')
    ax.plot(*line_pts(*BOUND['coef'][:2, c].astype(float), float(BOUND['coef'][2, c]), lim), '-.',
            color='#B03060', lw=1.8, zorder=3, label='gate $z=0$, planar fit')

    d = read[c].astype(float)
    d = d / np.linalg.norm(d) * lim * 0.78
    ax.annotate('', xy=d, xytext=(0, 0), zorder=4,
                arrowprops=dict(arrowstyle='-|>', lw=2.4, color=COL['h14'], shrinkA=0, shrinkB=0))
    ax.text(*(d * 1.16), BOUND_ARROW, color=COL['h14'], fontsize=TICK_FS - 1, ha='center',
            va='center', zorder=5,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.75))

    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_box_aspect(1)
    ax.set_title(f'rank {rk}   L{BOUND_LAYER} n{n}\n'
                 f'on {BOUND["on_frac"][c]:.0%}  slice {BOUND["acc_slice"][c]:.2f} / fit '
                 f'{BOUND["acc"][c]:.2f} vs base {BOUND_BASE[c]:.2f}  '
                 f'$R^2$={BOUND["r2"][c]:.2f}', fontsize=TICK_FS - 1)
    ax.tick_params(labelsize=TICK_FS - 3)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=8)
    if c % BOUND_NCOL == 0:
        ax.set_ylabel(r'$\mathbf{v}_2$ component', fontsize=LABEL_FS - 2)
    if c // BOUND_NCOL == nrow - 1:
        ax.set_xlabel(r'$\mathbf{v}_1$ component', fontsize=LABEL_FS - 2)
    if c == 0:
        ax.legend(fontsize=LEGEND_FS - 4, frameon=False, loc='upper left')
fig.suptitle(f'L{BOUND_LAYER} neurons in span($\\mathbf{{v}}_1,\\mathbf{{v}}_2$): activation, '
             f'{BOUND_ARROW} read direction, and where the neuron switches on',
             fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_l{BOUND_LAYER}_neuron_boundaries'); plt.show()

## Neurons that matter in both cases

The attribution ranking is computed separately per case, and the case *is* the perturbation:
`n1 larger` perturbs $y_1$, `n2 larger` perturbs $y_2$ (`make_batch` varies only the winning
slot's number). So intersecting the two top-$k$ lists picks out the neurons that matter **whichever
number is contested** — the case-independent part of the comparator.

For those neurons: their receptive fields over the $(y_1,y_2)$ grid, and the IIA when exactly that
set is frozen, against each case's own top-$n$ and random sets of the same size.

In [ ]:
SHARED_K   = 30                          # rank depth intersected in each case
SHARED_KS  = [10, 20, 30, 50, 100]       # depths reported in the overlap table
SHARED_PATH = f'{RESULTS_DIR}/comparator_shared_{REGIME}.npz'

# Rank position of every pooled neuron in each case, so lookups are O(1) rather than a list scan.
POS_IN = {}
for c in CASES:
    _p = np.empty(len(POOL_LAYER), dtype=np.int64)
    _p[ORDER[c]] = np.arange(len(POOL_LAYER))
    POS_IN[c] = _p

print(f'pool = {len(POOL_LAYER)} neurons, {len(POOL_LAYER) // len(NEURON_LAYERS)} per layer')
print(f'{"top-k each":>11}{"overlap":>9}{"chance":>9}{"fold":>8}{"L14/L15":>10}')
for k in SHARED_KS:
    s = np.array(sorted(set(ORDER[CASES[0]][:k]) & set(ORDER[CASES[1]][:k])), dtype=np.int64)
    ch = k * k / len(POOL_LAYER)
    print(f'{k:>11}{len(s):>9}{ch:>9.3f}{len(s) / ch:>7.0f}x'
          f'{f"{(POOL_LAYER[s] == 14).sum()}/{(POOL_LAYER[s] == 15).sum()}":>10}')

SHARED = np.array(sorted(set(ORDER[CASES[0]][:SHARED_K]) & set(ORDER[CASES[1]][:SHARED_K]),
                         key=lambda i: POS_IN[CASES[0]][i]), dtype=np.int64)
SH_LAYER, SH_WITHIN = POOL_LAYER[SHARED], POOL_WITHIN[SHARED]
print(f'\nthe top-{SHARED_K} overlap, {len(SHARED)} neurons (rank in {CASES[0]} / {CASES[1]}):')
for a in range(0, len(SHARED), 5):
    print('  ' + '  '.join(f'L{SH_LAYER[b]}n{SH_WITHIN[b]}'
                           f'({POS_IN[CASES[0]][SHARED[b]] + 1}/{POS_IN[CASES[1]][SHARED[b]] + 1})'
                           for b in range(a, min(a + 5, len(SHARED)))))


def _shared():
    out = {'shared_pooled': SHARED, 'shared_layer': SH_LAYER, 'shared_within': SH_WITHIN,
           'n_shared': np.array(len(SHARED)), 'shared_k': np.array(SHARED_K)}
    idx = {l: torch.as_tensor(SH_WITHIN[SH_LAYER == l], device=device, dtype=torch.long)
           for l in NEURON_LAYERS}

    # -- receptive fields, on the same grid the top-10 profile uses ---------------------------
    grid = np.arange(low + 1, high, RF_STEP)
    pairs = np.array([(a, b) for a in grid for b in grid], dtype=np.int64)
    ids = []
    for i0 in range(0, len(pairs), EVAL_BS):
        enc = tokenizer([D.prompt_gen(list(p)) for p in pairs[i0:i0 + EVAL_BS]],
                        return_tensors='pt')['input_ids']
        assert enc.shape[1] == T, 'every grid prompt must share the regime token layout'
        ids.append(enc.to(device))
    acts = capture_neurons(torch.cat(ids, dim=0), NEURON_LAYERS, POS_B, idx=idx)
    out['grid'] = grid.astype(np.int64)
    for l in NEURON_LAYERS:
        out[f'rf_acts_L{l}'] = acts[l].cpu().numpy().astype(np.float32)
        out[f'rf_idx_L{l}'] = idx[l].cpu().numpy()

    # -- freezing exactly this set, against same-size controls --------------------------------
    NEUR_CORR = {c: capture_neurons(BAT[c]['ids_r'], NEURON_LAYERS, POS_B) for c in CASES}
    n = len(SHARED)

    def run(case, pooled):
        sel = {l: torch.as_tensor(POOL_WITHIN[pooled[POOL_LAYER[pooled] == l]],
                                  device=device, dtype=torch.long) for l in NEURON_LAYERS}
        bat, recs, iias = BAT[case], [], []
        for lo in range(0, bat['n'], EVAL_BS):
            hi = min(lo + EVAL_BS, bat['n'])
            sub = _slice_batch(bat, lo, hi)
            handles = S.build_handles(ARMS[JOINT], SRC[case], lo, hi, POS_B)
            for l, ii in sel.items():
                if len(ii):
                    handles.append(mlp_of(l).down_proj.register_forward_pre_hook(
                        make_neuron_freeze_hook(ii, NEUR_CORR[case][l][lo:hi][:, ii], POS_B)))
            try:
                with torch.no_grad():
                    lg = D.model(sub['ids_r']).logits[:, -1, :].float()
            finally:
                for h in handles:
                    h.remove()
            r, i = posrec_and_iia(lg, sub)
            recs.append(r); iias.append(i)
            del lg
        return np.concatenate(recs), np.concatenate(iias)

    for case in CASES:
        sets = {'shared': SHARED, 'own': ORDER[case][:n]}
        for s in RAND_SEEDS:
            sets[f'rand{s}'] = np.random.default_rng(100 + s).choice(len(POOL_LAYER), n,
                                                                     replace=False)
        for nm, pooled in sets.items():
            rec, iia = run(case, np.asarray(pooled))
            out[f'posrec_{ck(case)}_{nm}'] = rec.astype(np.float32)
            out[f'iia_{ck(case)}_{nm}'] = iia.astype(np.float32)
    return out


SHR = cached_npz(SHARED_PATH, _shared)
if int(SHR['shared_k']) != SHARED_K:
    print(f'NOTE: the cache was built at SHARED_K={int(SHR["shared_k"])}, not {SHARED_K}. '
          'Delete the file to rebuild at the current depth.')

SH_SETS = ['shared', 'own'] + [f'rand{s}' for s in RAND_SEEDS]
print(f'\n--- IIA with {int(SHR["n_shared"])} neurons frozen at n2, joint patch (n={N_EVAL})')
print(f'{"case":>12}{"no freeze":>11}' + ''.join(f'{s:>10}' for s in SH_SETS) + f'{"all":>10}')
for case in CASES:
    print(f'{case:>12}{RES[case, JOINT, "none"][1].mean():11.3f}'
          + ''.join(f'{SHR[f"iia_{ck(case)}_{s}"].mean():10.3f}' for s in SH_SETS)
          + f'{TOPK[f"iia_{ck(case)}_all"].mean():10.3f}')

In [ ]:
SH_NCOL = 5

grid = SHR['grid']
ext = [grid.min(), grid.max(), grid.min(), grid.max()]
n_sh = len(SHR['shared_pooled'])
nrow = int(np.ceil(n_sh / SH_NCOL))

fig, axes = plt.subplots(nrow, SH_NCOL, figsize=(3.2 * SH_NCOL, 3.3 * nrow), squeeze=False)
for r in range(nrow * SH_NCOL):
    ax = axes[r // SH_NCOL, r % SH_NCOL]
    if r >= n_sh:
        ax.axis('off')
        continue
    l, wi = int(SHR['shared_layer'][r]), int(SHR['shared_within'][r])
    col = int(np.where(SHR[f'rf_idx_L{l}'] == wi)[0][0])
    a = SHR[f'rf_acts_L{l}'][:, col].reshape(len(grid), len(grid)).T   # pairs are y1-major
    v = np.abs(a).max()
    im = ax.imshow(a, origin='lower', extent=ext, aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v)
    ax.plot(ext[:2], ext[:2], '--', color='0.25', lw=1.0)
    p = SHR['shared_pooled'][r]
    ax.set_title(f'L{l} n{wi}   #{POS_IN[CASES[0]][p] + 1}/#{POS_IN[CASES[1]][p] + 1}',
                 fontsize=TICK_FS - 1)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=8)
    if r % SH_NCOL == 0:
        ax.set_ylabel('$y_2$', fontsize=LABEL_FS - 2)
    if r // SH_NCOL == nrow - 1:
        ax.set_xlabel('$y_1$', fontsize=LABEL_FS - 2)
    ax.tick_params(labelsize=TICK_FS - 3)
fig.suptitle(f'receptive fields of the {n_sh} neurons in the top-{SHARED_K} overlap  '
             f'(title: rank in {CASES[0]} / {CASES[1]}; dashed $y_1=y_2$)', fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_shared_receptive_fields'); plt.show()

In [ ]:
SH_LAB = {'shared': f'shared set ({int(SHR["n_shared"])})', 'own': "that case's own top-n",
          'rand0': 'random n', 'rand1': '_nolegend_'}
SH_COL = {'shared': COL['h14'], 'own': COL['full'], 'rand0': COL['ceil'], 'rand1': COL['ceil']}
xg = np.arange(len(SH_SETS))
Wg = 0.8 / len(CASES)

fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.2))
for ax, (metric, idx, ylab) in zip(axes, [('iia', 1, 'IIA'), ('posrec', 0, 'position recovery')]):
    for j, case in enumerate(CASES):
        m = np.array([SHR[f'{metric}_{ck(case)}_{s}'].mean() for s in SH_SETS])
        se = np.array([SHR[f'{metric}_{ck(case)}_{s}'].std() / np.sqrt(N_EVAL) for s in SH_SETS])
        b = ax.bar(xg + (j - (len(CASES) - 1) / 2) * Wg, m, Wg, yerr=se, capsize=3,
                   color=[COL['das'], COL['full']][j], alpha=0.9, edgecolor='white',
                   linewidth=0, label=case)
        ax.bar_label(b, fmt='%.2f', fontsize=TICK_FS - 3, padding=2)
        ax.axhline(RES[case, JOINT, 'none'][idx].mean(), ls='--', lw=1.1,
                   color=[COL['das'], COL['full']][j],
                   label=f'{case}, no freeze' if metric == 'iia' else None)
    ax.axhline(0, color='0.3', lw=0.8)
    ax.set_xticks(xg)
    ax.set_xticklabels([SH_LAB[s].replace('_nolegend_', 'random n') for s in SH_SETS],
                       fontsize=TICK_FS - 1, rotation=12, ha='right')
    style_axes(ax, ylabel=ylab)
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=LEGEND_FS - 2, frameon=False, loc='upper right')
fig.suptitle(f'freezing the {int(SHR["n_shared"])} neurons shared by both cases\' top '
             f'{SHARED_K}, under the joint patch', fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, f'{REGIME}_shared_freeze_iia'); plt.show()

## Save everything and check the manifest

Writes the few results that live only in the kernel (the displacement diagnostic, the sanity check,
the read-in SVD) and lists every result file. Once all of them are present, re-running the notebook
takes the plot-only path: no model, no captures, seconds to redraw.

In [ ]:
# Self-contained on purpose: this cell can be run on its own in a kernel that has only executed
# some of the notebook. Anything it needs that is missing is rebuilt here or read from disk, and
# whatever is not in memory is reported rather than silently skipped.
import os, json
import numpy as np

RESULTS_DIR = globals().get('RESULTS_DIR', 'results')
REGIME = globals().get('REGIME', '2digit')
_ck = globals().get('ck', lambda s: s.replace(' ', '_'))
CHECKS_PATH = f'{RESULTS_DIR}/comparator_checks_{REGIME}.npz'
if 'RESULT_FILES' not in globals():          # mirrors the list in the plot-style cell
    RESULT_FILES = {k: f'comparator_{k}{REGIME}.npz' for k in
                    ('', 'atp_', 'topk_', 'inject_', 'tune_', 'rf_', 'push_', 'mlpdas_', 'das15_',
                     'connect_', 'bound_', 'shared_')}

# Blocks that print rather than save. Merged into whatever the file already holds, so running this
# in a kernel that computed only some of them never erases the others.
extra = dict(np.load(CHECKS_PATH)) if os.path.exists(CHECKS_PATH) else {}
absent = []

if 'DISP' in globals():                      # cell 9, patch-source displacement
    for case, (du, dv) in DISP.items():
        extra[f'disp_{_ck(case)}'] = np.array([du, dv], dtype=np.float32)
elif not any(k.startswith('disp_') for k in extra):
    absent.append('DISP (cell 9)')

if 'SANITY' in globals():                    # cell 11, no-patch freeze
    for (case, tag), (rec, iia) in SANITY.items():
        extra[f'sanity_{_ck(case)}_{tag}'] = np.array([rec, iia], dtype=np.float32)
elif not any(k.startswith('sanity_') for k in extra):
    absent.append('SANITY (cell 11)')

if 'SVD' in globals():                       # cell 41, read-in SVD (pure numpy off the caches)
    for which, B in SVD.items():
        extra[f'svd_{which}_M'] = np.asarray(B['M'], dtype=np.float32)
        extra[f'svd_{which}_v1'] = np.asarray(B['v1'], dtype=np.float32)
        extra[f'svd_{which}_stats'] = np.array(
            [B['s1'], B['s2'], B['ratio'], B['ang'], B['dev'], B['pct'], B['pctdev']],
            dtype=np.float32)        # s1, s2, ratio, angle, deviation, pct, pct of deviation
elif not any(k.startswith('svd_') for k in extra):
    absent.append('SVD (cell 41)')

if extra:
    np.savez_compressed(CHECKS_PATH, **extra)
    print(f'wrote {CHECKS_PATH}   ({len(extra)} arrays)')
else:
    print('nothing to save yet')
if absent:
    print('not in this kernel and not already on disk: ' + ', '.join(absent)
          + '\n  -> set PLOT_ONLY = False and re-run those cells with the model to record them')

print(f'\n--- result manifest, {RESULTS_DIR}/')
print(f'{"file":>34}{"MB":>9}{"arrays":>9}   contents')
WHAT = {'': 'patch x freeze grid', 'atp_': 'per-neuron attribution',
        'topk_': 'top-k freeze curve', 'inject_': 'clean-MLP injection',
        'tune_': 'plane coords, tuning, alignment', 'rf_': 'receptive fields, read directions',
        'push_': 'the plane pushed through the neurons', 'mlpdas_': 'DAS in the neuron space',
        'das15_': 'DAS at the L15 residual', 'connect_': 'L15 alignment, L14->L15 edges',
        'bound_': 'gate preactivation and boundaries',
        'shared_': 'neurons shared by both cases: RFs and freeze'}
total, missing = 0.0, []
for key, fname in list(RESULT_FILES.items()) + [('checks_', os.path.basename(CHECKS_PATH))]:
    path = f'{RESULTS_DIR}/{fname}'
    what = WHAT.get(key, 'displacement, sanity, SVD (diagnostics, not needed for plots)')
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        total += mb
        with np.load(path) as z:
            n = len(z.files)
        print(f'{fname:>34}{mb:9.3f}{n:9d}   {what}')
    else:
        if key in RESULT_FILES:
            missing.append(fname)
        print(f'{fname:>34}{"--":>9}{"--":>9}   MISSING -- {what}')
mj = f'{RESULTS_DIR}/comparator_meta.json'
if os.path.exists(mj):
    total += os.path.getsize(mj) / 1e6
    print(f'{"comparator_meta.json":>34}{os.path.getsize(mj) / 1e6:9.3f}{"":>9}'
          f'   grid config and condition means')
print(f'{"TOTAL":>34}{total:9.3f}')
print('\nevery result is cached -- a fresh kernel will run plot-only' if not missing else
      f'\nstill to compute: {", ".join(missing)}')